# Barangay × category sales forecasting — standalone Colab workflow

Estimate **recorded sale units across participating stores** for the next **seven days**.
This is not total resident demand or an individual store's purchase recommendation.
Stockouts, changing store participation, and package mix affect the interpretation.

Start with `ML_WORKFLOW_GUIDE.md`. In Colab choose **Runtime → Run all** after editing
the configuration cell. The default `DEMO_PRESET="panabo_50k"` creates or loads 50,000 synthetic sale lines
for all 40 Panabo barangays, 80 fictional stores, and four categories. History is
May 13?November 8, 2026; the forecast covers November 9?15. Run all cells to use
the existing dashboard. No external dataset upload is needed for this preset.
For real inputs set `DEMO_PRESET=None`, then upload `transactions.csv` when requested. No application code,
price, customer identity, product name, or coordinates are required.

Modes: `train` evaluates, selects, fits, and forecasts; `forecast_only` reloads a trusted
artifact and uses newer compatible history. Demo sales and reporting history are **synthetic**, never evidence of real Panabo forecasting quality.
The city preset uses sourced approximate barangay reference points, not store locations or official centroids.

Includes a historical/forecast heat-map preview, independently evaluated store forecasts,
and notebook-only Random Forest/XGBoost/LightGBM comparison charts.


In [ ]:
# Install a compatible dependency set. The run records actual installed versions.
# This cell requires internet in a fresh Colab runtime. CPU is sufficient.
import importlib.util
import subprocess
import sys

DEPENDENCIES = {
    "numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.2,<3",
    "matplotlib": "matplotlib>=3.8,<4", "sklearn": "scikit-learn>=1.5,<2",
    "xgboost": "xgboost>=2.1,<4", "lightgbm": "lightgbm>=4.5,<5",
    "joblib": "joblib>=1.4,<2",
    "ipywidgets": "ipywidgets>=8,<9", "folium": "folium>=0.17,<1",
}
missing = [spec for module, spec in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import re
import time
import unicodedata
import uuid
import zipfile
import numpy as np
import pandas as pd

# ---------------------------- EDIT THESE SETTINGS ----------------------------
DEMO_PRESET = "panabo_50k"           # None = real inputs / legacy small demo
DEMO_DIRECTORY = Path("demo_data/panabo_50k")
MODE = "train"                       # "train" or "forecast_only"
CSV_PATH = Path("transactions.csv")
COVERAGE_PATH = None                  # e.g. Path("coverage.csv")
GENERATE_DEMO = False                 # True selects the legacy three-barangay demo
DATA_KIND = "real"                    # "real" or "synthetic"; demo sets synthetic
SOURCE_TIMEZONE = None                # e.g. "Asia/Manila" ONLY if source confirms naive times
AS_OF = None                         # aware timestamp; None = actual current time
FORECAST_CUTOFF = None                # YYYY-MM-DD; None = last complete Manila day

# Without coverage evidence, all four declarations are mandatory.
ASSUME_COMPLETE_REPORTING = False
REPORTING_START = None                # YYYY-MM-DD
REPORTING_END = None                  # YYYY-MM-DD
DECLARED_COHORT = []                  # [{"store_id":"s1","city":"Panabo",
                                     #   "barangay":"Salvacion","category_id":"c1"}]

# Keys are normalized automatically. Add reviewed misspellings here; no fuzzy merges.
CITY_ALIASES = {"Panabo City": "Panabo", "City of Panabo": "Panabo"}
BARANGAY_ALIASES = {}                 # {("Panabo", "Salvasion"): "Salvacion"}
# Optional reviewed registry: when populated, unlisted areas are quarantined.
CATEGORY_LABELS = {}                 # optional category_id -> display label
AREA_REGISTRY = []                    # [{"city":"Panabo","barangay":"Salvacion",
                                     #   "area_id":"application-area-id"}]
STATUS_ALIASES = {"completed": "completed", "cancelled": "cancelled",
                  "canceled": "cancelled", "voided": "voided", "refunded": "refunded"}
CHANNEL_ALIASES = {"pos": "pos", "online": "online"}
UNIT_ALIASES = {"pcs": "piece", "pc": "piece", "pieces": "piece",
                "bottles": "bottle", "packs": "pack"}
COORDINATES_PATH = None               # optional areas.csv; area_id,latitude,longitude,coordinate_source
SAVED_STORE_MODEL_PATH = None         # trusted store_method.joblib for forecast_only
SHOW_DASHBOARD = True                 # False for headless execution
SAVED_MODEL_PATH = None               # trusted method.joblib for forecast_only
OUTPUT_ROOT = Path("ml_outputs")
RANDOM_SEED = 42
FORECAST_DAYS = 7                     # fixed; longer horizons need another evaluation
MODEL_TREES = 120                     # reduce only for an explicitly labeled smoke run
SMOKE_RUN = False
# ---------------------------------------------------------------------------
REQUIRED_COLUMNS = [
    "transaction_datetime", "known_at", "transaction_id", "transaction_item_id",
    "transaction_status", "store_id", "city", "barangay", "product_id",
    "variant_id", "category_id", "quantity", "unit_label", "sales_channel",
]
GROUP = ["area_id", "category_id"]
STORE_GROUP = ["store_id", "area_id", "category_id"]
TREE_METHODS = ["random_forest", "xgboost", "lightgbm"]
ALL_CATEGORIES = "__all_categories__"
DASHBOARD_VERSION = "market-demand-preview-v1"
LAGS = [1, 7, 14, 28]
WINDOWS = [7, 14, 28]
TARGET_ID = "completed_gross_sale_units_v1"
FEATURE_VERSION = "sales-calendar-v1"
SCHEMA_VERSION = "market-demand-v1"
print("Mode:", MODE, "| horizon: 7 days | target:", TARGET_ID)


## Input contract and cleaning

Each row is a sale line, with the 15 configured columns. IDs must be stable strings.
`transaction_datetime` is the sale event; `known_at` is when this record was available.
Use offset timestamps; explicitly configure a source timezone for naive timestamps.
Completed sales count positively. Unsupported return adjustments need source reconciliation.
Refunded statuses are withheld rather than interpreted as returned quantities.

Area labels are normalized and reviewed aliases are applied within each city.
Without an area registry, normalized labels define **internal identities**, not official
government codes; possible spelling variations are listed for human review.
One pack counts as one recorded sale unit, with no inferred conversion.


In [ ]:
def normalized(value):
    if pd.isna(value):
        return ""
    return " ".join(unicodedata.normalize("NFKC", str(value)).strip().casefold().split())

def barangay_key(value):
    return re.sub(r"^(?:barangay|brgy\.?|brg\.?)\s+", "", normalized(value)).strip()

def canonical_area(city, barangay, config):
    city_map = {normalized(k): normalized(v) for k, v in config["city_aliases"].items()}
    c = city_map.get(normalized(city), normalized(city))
    b = barangay_key(barangay)
    aliases = {(city_map.get(normalized(x[0]), normalized(x[0])), barangay_key(x[1])):
               barangay_key(x[2]) for x in config["barangay_aliases"]}
    b = aliases.get((c, b), b)
    registry = {}
    for row in config["area_registry"]:
        rc = city_map.get(normalized(row["city"]), normalized(row["city"]))
        registry[(rc, barangay_key(row["barangay"]))] = str(row["area_id"])
    if not c or not b:
        return c, b, ""
    identity = registry.get((c, b), "") if registry else (
        "area-" + hashlib.sha256(json.dumps([c, b]).encode()).hexdigest()[:20])
    return c, b, identity

def parse_timestamp(value, source_timezone=None):
    try:
        t = pd.Timestamp(value)
        if pd.isna(t):
            return pd.NaT
        if t.tzinfo is None:
            if source_timezone is None:
                return pd.NaT
            t = t.tz_localize(source_timezone, ambiguous="raise", nonexistent="raise")
        return t.tz_convert("UTC")
    except (ValueError, TypeError):
        return pd.NaT

def canonical_locations(cities, barangays, config):
    # Resolve each distinct label pair once; city-wide exports repeat the same locations.
    keys = list(zip(map(normalized, cities), map(normalized, barangays)))
    lookup = {key:canonical_area(*key, config) for key in dict.fromkeys(keys)}
    return [lookup[key] for key in keys]

def clean_transactions(raw, config):
    missing = sorted(set(REQUIRED_COLUMNS) - set(raw.columns))
    if missing:
        raise ValueError("Missing required columns: " + ", ".join(missing))
    out = raw[REQUIRED_COLUMNS].copy().reset_index(drop=True)
    out["source_row"] = np.arange(2, len(out) + 2)
    for col in set(REQUIRED_COLUMNS) - {"quantity"}:
        out[col] = out[col].astype("string").str.strip()
    out["transaction_datetime"] = pd.to_datetime(out["transaction_datetime"].map(
        lambda v: parse_timestamp(v, config["source_timezone"])), utc=True)
    out["known_at"] = pd.to_datetime(out["known_at"].map(
        lambda v: parse_timestamp(v, config["source_timezone"])), utc=True)
    out["quantity"] = pd.to_numeric(out["quantity"], errors="coerce")
    for col, mapping in [("transaction_status", config["status_aliases"]),
                         ("sales_channel", config["channel_aliases"])]:
        lookup = {normalized(k): v for k, v in mapping.items()}
        out[col] = out[col].map(normalized).map(lookup).fillna("")
    units = {normalized(k): normalized(v) for k, v in config["unit_aliases"].items()}
    out["unit_label"] = out["unit_label"].map(lambda v: units.get(normalized(v), normalized(v)))
    locations = canonical_locations(out.city, out.barangay, config)
    out[["city", "barangay", "area_id"]] = pd.DataFrame(locations, index=out.index)
    reasons = pd.Series("", index=out.index, dtype="string")
    def reject(mask, reason):
        mask = mask.fillna(True)
        reasons.loc[mask] = reasons.loc[mask] + reason + ";"
    for col in ["transaction_id", "transaction_item_id", "store_id", "product_id",
                "variant_id", "category_id", "unit_label", "area_id"]:
        reject(out[col].isna() | out[col].eq(""), "invalid_" + col)
    reject(out.transaction_datetime.isna(), "invalid_event_time")
    reject(out.known_at.isna(), "invalid_known_at")
    reject(out.known_at < out.transaction_datetime, "known_before_event")
    reject(out.transaction_status.eq(""), "unsupported_status")
    reject(out.transaction_status.eq("refunded"), "refund_requires_reconciliation")
    reject(out.sales_channel.eq(""), "unsupported_channel")
    reject(~np.isfinite(out.quantity) | out.quantity.le(0), "invalid_quantity")
    # Do not arbitrarily choose one conflicting version; snapshot history is not an event log.
    identity = ["sales_channel", "store_id", "transaction_id", "transaction_item_id"]
    facts = [c for c in out.columns if c != "source_row"]
    exact = out.duplicated(subset=facts, keep="first")
    distinct = out.loc[~exact]
    conflicting_keys = distinct.loc[distinct.duplicated(identity, keep=False), identity].drop_duplicates()
    conflicts = out[identity].merge(conflicting_keys.assign(conflict=True), how="left")["conflict"].fillna(False).astype(bool)
    reject(conflicts, "conflicting_duplicate")
    rejected = out.loc[reasons.ne("")].copy()
    rejected["reason"] = reasons.loc[rejected.index]
    duplicate_rows = out.loc[exact & reasons.eq("")].copy()
    valid = out.loc[reasons.eq("") & ~exact].copy()
    valid["date"] = valid.transaction_datetime.dt.tz_convert("Asia/Manila").dt.tz_localize(None).dt.normalize()
    completed = valid.loc[valid.transaction_status.eq("completed")].copy()
    # Invalid records that cannot be located make the whole input unsafe to aggregate.
    global_block = bool(len(rejected) and (
        rejected["area_id"].eq("").any() or rejected.category_id.isna().any()
        or rejected.category_id.eq("").any()))
    blocked = rejected.loc[rejected.area_id.ne("") & rejected.category_id.notna(),
                           ["store_id", *GROUP, "known_at", "transaction_datetime"]].drop_duplicates()
    audit = {"input_rows": len(out), "accepted_completed_rows": len(completed),
             "excluded_status_rows": len(valid) - len(completed),
             "exact_duplicates_removed": len(duplicate_rows), "quarantined_rows": len(rejected),
             "quarantined_measurable_quantity": float(rejected.quantity.where(
                 np.isfinite(rejected.quantity)).sum()), "global_block": global_block}
    return completed, rejected, duplicate_rows, blocked, audit

def current_config():
    return {"source_timezone": SOURCE_TIMEZONE, "city_aliases": CITY_ALIASES,
            "barangay_aliases": [[c, b, v] for (c, b), v in BARANGAY_ALIASES.items()],
            "area_registry": AREA_REGISTRY, "status_aliases": STATUS_ALIASES,
            "channel_aliases": CHANNEL_ALIASES, "unit_aliases": UNIT_ALIASES}

def choose_input(path):
    path = Path(path)
    if path.exists():
        return path
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Place {path} beside the notebook or update its path.")
    print(f"Upload {path.name} (and optional sidecar files) now.")
    files.upload()
    if not path.exists():
        raise FileNotFoundError(f"Expected {path.name}; update CSV_PATH if named differently.")
    return path

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def json_write(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False, default=str), encoding="utf-8")


## Synthetic data profiles
The default `DEMO_PRESET="panabo_50k"` creates 50,000 sale lines for all 40 Panabo
barangays, two fictional stores each, four categories, and 180 complete dates through
November 8, 2026. The generator and sourced geographic reference snapshot are embedded
below, so this notebook also works by itself in Colab. Existing files are checksum-checked.

`coverage.csv` contains 57,600 fully reported store/category/day statements. Reference
CSVs describe barangays, stores, products, and consistent package sizes. Two supplemental
daily-observation CSVs make the data easy to inspect outside the notebook; training
recomputes observations from the sale lines and coverage instead of trusting these summaries.

Weekly effects, trends, store differences, and sale quantities are assumed, not measured.
Each store/category has a reserved positive sale every four dates to keep the demo usable;
confirmed no-sale days are zero. Stock availability remains unknown. Category IDs are
readable synthetic identifiers, not application catalog IDs. All results stay labeled synthetic.

The smaller existing demo is still available with `DEMO_PRESET=None` and `GENERATE_DEMO=True`:
three barangays, nine stores, two categories. Its map points are illustrative. For real data
set `DEMO_PRESET=None` and `GENERATE_DEMO=False` and configure your input paths.


In [ ]:
def generate_demo(directory, end_date, days=180, seed=42):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    end = pd.Timestamp(end_date).normalize()
    dates = pd.date_range(end - pd.Timedelta(days=days-1), end)
    sales, coverage = [], []
    areas = ["Salvacion", "Gredu", "New Pandan"]
    for day_index, date in enumerate(dates):
        known = (date + pd.Timedelta(days=1, minutes=5)).tz_localize("Asia/Manila").isoformat()
        for a, area in enumerate(areas):
            for s in range(3):
                store = f"demo-store-{a}-{s}"
                for k in range(2):
                    cat = f"demo-category-{k}"
                    coverage.append(dict(store_id=store, city="Panabo", barangay=area,
                        category_id=cat, date=date.date().isoformat(), participation_status="active",
                        reporting_status="complete", operating_status="open",
                        availability_status="unknown", known_at=known, coverage_source="synthetic_generator"))
                    expected = (4 + a + k * 2 + s) * (1.35 if date.dayofweek >= 5 else 1) * (1 + day_index/900)
                    quantity = int(rng.poisson(expected))
                    if not quantity:
                        continue
                    tx = f"demo:{day_index}:{a}:{s}:{k}"
                    sales.append(dict(transaction_datetime=(date + pd.Timedelta(hours=12)).tz_localize("Asia/Manila").isoformat(),
                        known_at=known, transaction_id=tx, transaction_item_id=tx+":line:1",
                        transaction_status="COMPLETED", store_id=store, city="Panabo City",
                        barangay=[area, "Brgy. " + area, "Barangay " + area][s],
                        product_id=f"demo-product-{k}", variant_id=f"demo-variant-{k}",
                        category_id=cat, quantity=quantity, unit_label="bottle" if k == 0 else "pack",
                        sales_channel="POS"))
    transaction_path, coverage_path = directory/"transactions.csv", directory/"coverage.csv"
    pd.DataFrame(sales, columns=REQUIRED_COLUMNS).to_csv(transaction_path, index=False)
    pd.DataFrame(coverage).to_csv(coverage_path, index=False)
    json_write(directory/"dataset_manifest.json", {"data_kind":"synthetic", "seed":seed,
        "start":str(dates.min().date()), "end":str(end.date()), "days":days,
        "transactions_sha256":sha256_file(transaction_path), "coverage_sha256":sha256_file(coverage_path)})
    return transaction_path, coverage_path


In [ ]:
# Generated by scripts/sync_panabo_notebook.py; edit the canonical script/reference.
"""Reproducible synthetic Panabo sale lines. Uses only Python's standard library.

Run from the project root: python scripts/generate_panabo.py
The notebook embeds the same generator and reference snapshot for standalone Colab use.
"""
import argparse
from collections import Counter
import csv
from datetime import date, datetime, time as datetime_time, timedelta, timezone
import hashlib
import json
from pathlib import Path
import random


PANABO_COLUMNS = [
    "transaction_datetime", "known_at", "transaction_id", "transaction_item_id",
    "transaction_status", "store_id", "city", "barangay", "product_id",
    "variant_id", "category_id", "quantity", "unit_label", "sales_channel",
]
PANABO_CATEGORIES = [
    ("demo-beverages", "Beverages", "bottle", "500 mL bottle", 1.25),
    ("demo-snacks", "Snacks", "pack", "30 g pack", 1.10),
    ("demo-canned-goods", "Canned goods", "can", "155 g can", 0.85),
    ("demo-instant-noodles", "Instant noodles", "pack", "55 g pack", 1.00),
]
PANABO_GENERATOR_VERSION = "panabo-synthetic-v1"
PANABO_TZ = timezone(timedelta(hours=8))


def panabo_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def panabo_write_csv(path, columns, rows):
    with Path(path).open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def generate_panabo(directory, end_date="2026-11-08", days=180, rows=50000,
                    seed=42, reference=None):
    """Generate exact-count sale lines plus complete coverage and reference files.

    A rotating one-in-four-day sale floor gives every series seven positive dates
    in every complete 28-day window. Remaining sale lines are weighted draws.
    This floor is an explicit demo assumption, never inferred reporting evidence.
    """
    end = date.fromisoformat(str(end_date))
    if not isinstance(days, int) or days < 84:
        raise ValueError("Use at least 84 complete dates for notebook evaluation.")
    if not isinstance(rows, int) or rows <= 0:
        raise ValueError("rows must be a positive integer")
    if reference is None:
        source = Path(__file__).resolve().parents[1]/"data"/"panabo_reference.json"
        reference = json.loads(source.read_text(encoding="utf-8"))
    areas = reference["areas"]
    if len(areas) != 40 or len({a["area_id"] for a in areas}) != 40:
        raise ValueError("The Panabo profile requires 40 unique barangays.")
    start = end - timedelta(days=days-1)
    dates = [start + timedelta(days=i) for i in range(days)]
    rng = random.Random(seed)
    stores, slots, weights, counts = [], [], [], []
    for a, area in enumerate(areas):
        # Arbitrary synthetic effects, independent of population and geography.
        area_effect = rng.uniform(0.75, 1.35)
        for s in range(2):
            store = {"store_id":f"demo-panabo-{area['area_id']}-{s+1:02d}",
                     "store_name":f"Demo {area['barangay']} Store {s+1}",
                     "area_id":area["area_id"], "city":"Panabo",
                     "barangay":area["barangay"], "data_kind":"synthetic"}
            stores.append(store)
            store_effect = area_effect * rng.uniform(0.8, 1.2)
            for k, category in enumerate(PANABO_CATEGORIES):
                phase = rng.randrange(4)
                trend = rng.uniform(-0.08, 0.18)
                for d, day in enumerate(dates):
                    weekend = 1.25 if day.weekday() >= 5 else 1.0
                    weight = store_effect * category[4] * weekend * (1 + trend*d/(days-1))
                    slots.append((store, k, d))
                    weights.append(weight)
                    counts.append(int(d % 4 == phase))
    floor = sum(counts)
    if rows < floor:
        raise ValueError(f"Need at least {floor} rows for this profile's activity floor; requested {rows}.")
    cumulative, total = [], 0.0
    for weight in weights:
        total += weight
        cumulative.append(total)
    for index in rng.choices(range(len(slots)), cum_weights=cumulative, k=rows-floor):
        counts[index] += 1
    sales, coverage, store_daily, area_daily = [], [], [], Counter()
    for (store, k, d), count, weight in zip(slots, counts, weights):
        day = dates[d]
        category_id, _, unit, _, _ = PANABO_CATEGORIES[k]
        known = datetime.combine(day+timedelta(days=1), datetime_time(0, 5), PANABO_TZ).isoformat()
        coverage.append(dict(store_id=store["store_id"], city="Panabo", barangay=store["barangay"],
            category_id=category_id, date=day.isoformat(), participation_status="active",
            reporting_status="complete", operating_status="open", availability_status="unknown",
            known_at=known, coverage_source=PANABO_GENERATOR_VERSION))
        sold = 0
        for line in range(count):
            # Two fictional products per category share the same package size.
            product = f"{category_id}-product-{rng.randrange(2)+1:02d}"
            quantity = max(1, min(15, int(round(rng.gammavariate(2.2, weight/1.7)))))
            sold += quantity
            event = datetime.combine(day, datetime_time(8, 0), PANABO_TZ) + timedelta(seconds=rng.randrange(14*3600))
            identity = f"demo-panabo:{day.isoformat()}:{store['store_id']}:{k}:{line+1}"
            sales.append(dict(transaction_datetime=event.isoformat(), known_at=known,
                transaction_id=identity, transaction_item_id=identity+":line:1",
                transaction_status="completed", store_id=store["store_id"], city="Panabo",
                barangay=store["barangay"], product_id=product, variant_id=product+"-standard",
                category_id=category_id, quantity=quantity, unit_label=unit,
                sales_channel="online" if rng.random() < 0.18 else "pos"))
        store_daily.append(dict(store_id=store["store_id"], area_id=store["area_id"],
            category_id=category_id, date=day.isoformat(), units_sold=sold,
            observation_status="complete", data_kind="synthetic"))
        area_daily[(store["area_id"], category_id, day.isoformat())] += sold
    sales.sort(key=lambda r:(r["transaction_datetime"], r["transaction_item_id"]))
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    area_columns = ["area_id", "city", "barangay", "latitude", "longitude", "coordinate_source",
                    "coordinate_source_url", "identity_source_url", "source_checked_on"]
    categories = [dict(category_id=c[0], category_label=c[1], unit_label=c[2], package_size=c[3],
                       unit_policy="recorded_sale_units", data_kind="synthetic") for c in PANABO_CATEGORIES]
    products = [dict(product_id=f"{c[0]}-product-{p:02d}", variant_id=f"{c[0]}-product-{p:02d}-standard",
                     product_name=f"Demo {c[1]} {p}", category_id=c[0], unit_label=c[2],
                     package_size=c[3], data_kind="synthetic") for c in PANABO_CATEGORIES for p in (1, 2)]
    tables = {
        "transactions.csv":(PANABO_COLUMNS, sales),
        "coverage.csv":(list(coverage[0]), coverage),
        "areas.csv":(area_columns, [{key:a.get(key, "") for key in area_columns} for a in areas]),
        "stores.csv":(list(stores[0]), stores),
        "categories.csv":(list(categories[0]), categories),
        "products.csv":(list(products[0]), products),
        "store_daily_observations.csv":(list(store_daily[0]), store_daily),
        "area_daily_observations.csv":(["area_id", "category_id", "date", "units_sold", "observation_status", "data_kind"],
            [dict(area_id=a, category_id=c, date=d, units_sold=q, observation_status="complete", data_kind="synthetic")
             for (a,c,d), q in sorted(area_daily.items())]),
    }
    for filename, (columns, records) in tables.items():
        panabo_write_csv(directory/filename, columns, records)
    checksums = {name:panabo_sha256(directory/name) for name in sorted(tables)}
    manifest = {
        "schema_version":"market-demand-v1", "generator_version":PANABO_GENERATOR_VERSION,
        "data_kind":"synthetic", "seed":seed, "start":start.isoformat(), "end":end.isoformat(),
        "days":days, "rows":rows, "area_count":40, "store_count":80, "category_count":4,
        "forecast_start":(end+timedelta(days=1)).isoformat(), "forecast_end":(end+timedelta(days=7)).isoformat(),
        "as_of":datetime.combine(end+timedelta(days=1), datetime_time(0, 15), PANABO_TZ).isoformat(),
        "row_counts":{name:len(table[1]) for name, table in tables.items()},
        "total_recorded_units":sum(r["quantity"] for r in sales),
        "reference_version":reference["reference_version"],
        "reference_sha256":hashlib.sha256(json.dumps(reference, sort_keys=True, ensure_ascii=False).encode()).hexdigest(),
        "identity_source_url":areas[0]["identity_source_url"],
        "identity_reference_date":reference["identity_reference_date"],
        "coordinate_policy":"Published approximate barangay reference points; not official centroids or store coordinates. Unverified points stay blank.",
        "assumptions":[
            "All sales, stores, products, channels and reporting history are simulated; geography is sourced separately.",
            "Fixed cohort: two stores in every barangay, all four categories, open and completely reported on every date.",
            "Availability is unknown. No-sale dates with complete reporting are zero realized sales.",
            "One positive-sale date every four dates per store/category is reserved for demo eligibility; remaining lines use weighted sampling.",
            "Arbitrary area/store effects, 25% weekend weight increase, category differences and -8% to +18% series trends are synthetic assumptions.",
            "Quantities are bounded gamma draws; 18% online-channel probability; each product has one standard package size.",
            "Records and coverage become known at 00:05 Manila the following day; forecast issuance is 00:15.",
            "Combined category units count packages of different types, not comparable physical consumption.",
            "Synthetic evaluation demonstrates code behavior, not actual Panabo demand or real-world model accuracy.",
        ],
        "files":checksums,
        "dataset_fingerprint":hashlib.sha256(json.dumps(checksums, sort_keys=True).encode()).hexdigest(),
    }
    (directory/"dataset_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")
    return directory/"transactions.csv", directory/"coverage.csv"


def panabo_preset(directory, reference):
    """Create missing demo input; validate existing inputs before configuring a run."""
    directory = Path(directory)
    manifest_path = directory/"dataset_manifest.json"
    if not manifest_path.exists():
        generate_panabo(directory, reference=reference)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("data_kind") != "synthetic" or manifest.get("generator_version") != PANABO_GENERATOR_VERSION:
        raise ValueError("Incompatible Panabo demo manifest; regenerate with scripts/generate_panabo.py.")
    for filename, checksum in manifest["files"].items():
        if filename not in {"transactions.csv", "coverage.csv", "areas.csv", "stores.csv", "categories.csv", "products.csv",
                            "store_daily_observations.csv", "area_daily_observations.csv"}:
            raise ValueError("Unexpected Panabo demo artifact.")
        if panabo_sha256(directory/filename) != checksum:
            raise ValueError(f"Panabo demo checksum mismatch: {filename}; regenerate the dataset.")
    required = {"transactions.csv", "coverage.csv", "areas.csv"}
    if not required.issubset(manifest["files"]):
        raise ValueError("Panabo demo manifest is missing required inputs.")
    with (directory/"areas.csv").open(encoding="utf-8", newline="") as stream:
        registry = [{k:r[k] for k in ("city", "barangay", "area_id")} for r in csv.DictReader(stream)]
    return {"csv_path":directory/"transactions.csv", "coverage_path":directory/"coverage.csv",
            "coordinates_path":directory/"areas.csv", "area_registry":registry,
            "cutoff":manifest["end"], "as_of":manifest["as_of"], "manifest":manifest}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--output", type=Path, default=Path("demo_data/panabo_50k"))
    parser.add_argument("--end-date", default="2026-11-08")
    parser.add_argument("--days", type=int, default=180)
    parser.add_argument("--rows", type=int, default=50000)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    paths = generate_panabo(args.output, args.end_date, args.days, args.rows, args.seed)
    print(f"Generated {args.rows:,} synthetic sale lines: {paths[0]}")
    print(f"Coverage: {paths[1]}")

PANABO_REFERENCE = {'reference_version': 'panabo-reference-v1', 'identity_reference_date': '2025-07-31', 'notes': 'PSA city directory displays a 2025-07-31 roster. Reference points are approximate published locations, not surveyed boundaries. No population or coordinates are used to infer actual sales.', 'areas': [{'area_id': '1102315001', 'city': 'Panabo', 'barangay': 'A. O. Floirendo', 'latitude': '7.3977', 'longitude': '125.5802', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/a-o-floirendo.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315002', 'city': 'Panabo', 'barangay': 'Datu Abdul Dadia', 'latitude': '7.3153', 'longitude': '125.6548', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/datu-abdul-dadia.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315003', 'city': 'Panabo', 'barangay': 'Buenavista', 'latitude': '7.2756', 'longitude': '125.5907', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/buenavista.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315004', 'city': 'Panabo', 'barangay': 'Cacao', 'latitude': '7.3083', 'longitude': '125.6077', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/cacao.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315005', 'city': 'Panabo', 'barangay': 'Cagangohan', 'latitude': '7.2815', 'longitude': '125.6829', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/cagangohan.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315006', 'city': 'Panabo', 'barangay': 'Consolacion', 'latitude': '7.3169', 'longitude': '125.5538', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/consolacion.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315007', 'city': 'Panabo', 'barangay': 'Dapco', 'latitude': '7.3921', 'longitude': '125.5983', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/dapco.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315009', 'city': 'Panabo', 'barangay': 'Gredu', 'latitude': '7.2957', 'longitude': '125.6776', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/gredu.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315010', 'city': 'Panabo', 'barangay': 'J.P. Laurel', 'latitude': '7.2759', 'longitude': '125.6700', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/j-p-laurel.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315011', 'city': 'Panabo', 'barangay': 'Kasilak', 'latitude': '7.3268', 'longitude': '125.5951', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/kasilak.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315012', 'city': 'Panabo', 'barangay': 'Katipunan', 'latitude': '7.3007', 'longitude': '125.6306', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/katipunan.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315013', 'city': 'Panabo', 'barangay': 'Katualan', 'latitude': '7.2301', 'longitude': '125.5543', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/katualan.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315014', 'city': 'Panabo', 'barangay': 'Kauswagan', 'latitude': '7.3102', 'longitude': '125.5831', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/kauswagan.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315015', 'city': 'Panabo', 'barangay': 'Kiotoy', 'latitude': '7.2443', 'longitude': '125.6077', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/kiotoy.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315016', 'city': 'Panabo', 'barangay': 'Little Panay', 'latitude': '7.2979', 'longitude': '125.6482', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/little-panay.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315017', 'city': 'Panabo', 'barangay': 'Lower Panaga', 'latitude': '7.4320', 'longitude': '125.5640', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/lower-panaga.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315018', 'city': 'Panabo', 'barangay': 'Mabunao', 'latitude': '7.2543', 'longitude': '125.5745', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/mabunao.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315019', 'city': 'Panabo', 'barangay': 'Maduao', 'latitude': '7.2796', 'longitude': '125.6433', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/maduao.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315020', 'city': 'Panabo', 'barangay': 'Malativas', 'latitude': '7.2936', 'longitude': '125.5648', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/malativas.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315021', 'city': 'Panabo', 'barangay': 'Manay', 'latitude': '7.3456', 'longitude': '125.6022', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/manay.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315022', 'city': 'Panabo', 'barangay': 'Nanyo', 'latitude': '7.3329', 'longitude': '125.6361', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/nanyo.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315023', 'city': 'Panabo', 'barangay': 'New Malaga', 'latitude': '7.3442', 'longitude': '125.5725', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/new-malaga.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315024', 'city': 'Panabo', 'barangay': 'New Malitbog', 'latitude': '7.3339', 'longitude': '125.6209', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/new-malitbog.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315025', 'city': 'Panabo', 'barangay': 'New Pandan', 'latitude': '7.2973', 'longitude': '125.6801', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/new-pandan.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315026', 'city': 'Panabo', 'barangay': 'New Visayas', 'latitude': '7.3081', 'longitude': '125.6682', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/new-visayas.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315027', 'city': 'Panabo', 'barangay': 'Quezon', 'latitude': '7.3327', 'longitude': '125.6795', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/quezon.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315028', 'city': 'Panabo', 'barangay': 'Salvacion', 'latitude': '7.3182', 'longitude': '125.6882', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/salvacion.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315029', 'city': 'Panabo', 'barangay': 'San Francisco', 'latitude': '7.3068', 'longitude': '125.6803', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/san-francisco.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315030', 'city': 'Panabo', 'barangay': 'San Nicolas', 'latitude': '7.2626', 'longitude': '125.6181', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/san-nicolas.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315031', 'city': 'Panabo', 'barangay': 'San Roque', 'latitude': '7.2552', 'longitude': '125.5533', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/san-roque.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315032', 'city': 'Panabo', 'barangay': 'San Vicente', 'latitude': '7.3088', 'longitude': '125.7003', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/san-vicente.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315033', 'city': 'Panabo', 'barangay': 'Santa Cruz', 'latitude': '7.2365', 'longitude': '125.5896', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/santa-cruz.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315034', 'city': 'Panabo', 'barangay': 'Santo Niño', 'latitude': '7.3082', 'longitude': '125.6867', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/santo-nino.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315035', 'city': 'Panabo', 'barangay': 'Sindaton', 'latitude': '7.4396', 'longitude': '125.5842', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/sindaton.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315036', 'city': 'Panabo', 'barangay': 'Southern Davao', 'latitude': '7.3323', 'longitude': '125.6577', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/southern-davao.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315037', 'city': 'Panabo', 'barangay': 'Tagpore', 'latitude': '7.2743', 'longitude': '125.6250', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/tagpore.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315038', 'city': 'Panabo', 'barangay': 'Tibungol', 'latitude': '7.3947', 'longitude': '125.5555', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/tibungol.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315039', 'city': 'Panabo', 'barangay': 'Upper Licanan', 'latitude': '7.2856', 'longitude': '125.6325', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/upper-licanan.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315040', 'city': 'Panabo', 'barangay': 'Waterfall', 'latitude': '7.2886', 'longitude': '125.5834', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/waterfall.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}, {'area_id': '1102315041', 'city': 'Panabo', 'barangay': 'San Pedro', 'latitude': '7.2973', 'longitude': '125.7106', 'coordinate_source': 'PhilAtlas approximate barangay reference; not a store location or official centroid', 'coordinate_source_url': 'https://www.philatlas.com/mindanao/r11/davao-del-norte/panabo/san-pedro.html', 'identity_source_url': 'https://psa.gov.ph/classification/psgc/barangays/1102315000', 'source_checked_on': '2026-11-09'}]}


In [ ]:
# Generated by scripts/sync_panabo_notebook.py; edit scripts/notebook_guidance.py.
"""Plain-language notebook helpers, embedded by sync_panabo_notebook.py.

The embedded cell uses the notebook's pandas, numpy and configuration globals.
Keep these helpers independent of files so downloaded notebooks remain standalone.
"""


def method_label(method):
    return {"last_week": "Repeat last week's sales", "weekday_average": "Average of the last four matching weekdays",
            "random_forest": "Random Forest", "xgboost": "XGBoost", "lightgbm": "LightGBM"}.get(method, str(method))


def method_chart_label(method):
    return {"last_week":"Repeat last week", "weekday_average":"Four-week weekday average"}.get(method, method_label(method))


def category_label(category):
    if str(category) == ALL_CATEGORIES:
        return "All categories (mixed sale units)"
    labels = {str(k): str(v) for k, v in CATEGORY_LABELS.items()}
    return labels.get(str(category), f"Category {category} - name unavailable")


def readable_table(frame, labels=None):
    """Add display labels without changing the source table or exported schema."""
    result = frame.copy()
    if "category_id" in result:
        result.insert(result.columns.get_loc("category_id") + 1, "category_name", result.category_id.map(category_label))
    if "area_id" in result and labels:
        result.insert(result.columns.get_loc("area_id") + 1, "area_name", result.area_id.map(lambda a: labels.get(a, str(a))))
    if "method" in result:
        result["method"] = result.method.map(method_label)
    if "status" in result:
        meanings = {"eligible":"Ready to forecast", "insufficient_history":"Fewer than 56 completely reported dates",
                    "incomplete_reporting":"Some of the latest 28 dates have unknown sales",
                    "no_sales":"No recorded sales in the latest 28 dates",
                    "sparse_sales":"Fewer than seven sales-active dates in the latest 28 dates",
                    "new_group_or_changed_cohort_requires_retraining":"New combination or changed participating stores: retrain first",
                    "insufficient_store_evaluation":"Not enough comparable past tests for this store",
                    "saved_store_model_unavailable":"No saved store model was supplied"}
        result["status"] = result.status.map(lambda s: meanings.get(s, str(s).replace("_", " ")))
    if "predicted_units" in result:
        result["predicted_units"] = result.predicted_units.round(2)
    for column in ["lower_bound", "upper_bound"]:
        if column in result:
            result[column] = result[column].astype(object).where(result[column].notna(), "Unavailable")
    return result


def show_markdown(text):
    from IPython.display import Markdown, display
    display(Markdown(text))


METRIC_GUIDE = """How to read the scores (one prediction means one category in one area or store on one date):

- MAE: average number of units the daily estimate missed by. Lower is better; this chooses the method.
- RMSE: another error measure in units, giving larger misses more weight. Lower is better.
- bias: average prediction minus recorded sales. Negative means estimates tend to be too low; positive means too high. Closer to zero is better.
- WAPE_pct: total absolute error divided by total recorded sales, as a percentage. Lower is better. This is not percentage accuracy; unavailable when recorded sales total zero.
- rows: number of daily predictions checked, not the number of original sale lines.

These scores summarize past tests; they are not a guaranteed error range for a future prediction."""


def cleaning_explanation(audit):
    return (f"Read {audit['input_rows']:,} sale lines; kept {audit['accepted_completed_rows']:,} completed-sale lines. "
            f"Removed {audit['exact_duplicates_removed']:,} exact duplicates, excluded {audit['excluded_status_rows']:,} "
            f"lines with other sale statuses, and set aside {audit['quarantined_rows']:,} lines needing review. "
            + ("Unlocatable rejected records block the run until repaired." if audit['global_block'] else
               "No unlocatable rejected records block the whole run. Affected combinations may still be withheld."))


def validation_explanation(comparison):
    if comparison.empty:
        return "No comparable validation results are available; a best method cannot be declared."
    ordered = comparison.sort_values(["MAE", "tie_priority"] if "tie_priority" in comparison else ["MAE"], kind="stable")
    winner = ordered.iloc[0]
    rmse_best = comparison.loc[comparison.RMSE.eq(comparison.RMSE.min())]
    names = ", ".join(method_label(m) for m in rmse_best.method)
    text = (f"Validation choice: {method_label(winner.method)}. Its daily estimates missed by {winner.MAE:.2f} units on average (MAE). "
            f"Lowest RMSE: {names}, at {rmse_best.iloc[0].RMSE:.2f} units. ")
    if not rmse_best.method.eq(winner.method).any():
        text += "The measures favor different methods: MAE measures average absolute error; RMSE gives larger misses more weight. We choose by MAE, so you do not need to choose between the charts. "
    else:
        text += "The MAE choice also has the lowest RMSE. "
    if comparison.MAE.eq(winner.MAE).sum() > 1:
        text += "An exact MAE tie was resolved by the existing fixed method order (simple methods first). "
    text += "Selection uses full-precision scores; rounded displays can look tied. This choice covers all eligible combinations in this scope, not each category separately."
    return text


def evaluation_explanation(evaluation, method):
    if evaluation.get("mode") == "forecast_only":
        original = evaluation.get("training_evaluation", {})
        old = next((r for r in original.get("final", []) if r["method"] == method), None)
        detail = f" Its saved final-check MAE was {old['MAE']:.2f} units." if old else ""
        trained = evaluation.get("model_training_end")
        training_text = f" The saved model was trained using history through {trained}." if trained else ""
        return (f"Using the saved {method_label(method)} method to refresh the forecast. No new model comparison or accuracy measurement was performed."
                + training_text
                + detail + " Saved scores describe its earlier evaluation, not accuracy on the new forecast dates.")
    records = evaluation.get("validation", [])
    lines = [validation_explanation(pd.DataFrame(records))] if records else []
    selected = evaluation.get("validation_selected_method")
    baseline = evaluation.get("strongest_validation_baseline")
    final = {r["method"]: r for r in evaluation.get("final", [])}
    if method not in final:
        return "\n\n".join(lines + [evaluation.get("message", "No final-check results are available; no evaluated method can be recommended.")])
    lines.append(f"Method used for the forecast: {method_label(method)}. On the later final-check weeks, its daily estimates missed by {final[method]['MAE']:.2f} units on average.")
    measured = final[method]
    if measured.get("bias") is not None:
        bias = measured["bias"]
        lines.append(f"Its predictions averaged {abs(bias):.2f} units {'below' if bias < 0 else 'above'} recorded sales (signed bias {bias:+.2f}). This is an average tendency, not a correction to apply to every forecast.")
    if "WAPE_pct" in measured:
        wape = measured["WAPE_pct"]
        lines.append(f"Total absolute error was {wape:.2f}% of total recorded sales (WAPE); this is not an accuracy percentage." if wape is not None else
                     "WAPE is unavailable because the recorded sales total was zero.")
    if measured.get("rows") is not None:
        lines.append(f"These scores cover {measured['rows']:,} checked daily category predictions, not original sale lines.")
    if selected in final and baseline in final and selected != baseline:
        difference = final[baseline]["MAE"] - final[selected]["MAE"]
        base_error = final[baseline]["MAE"]
        percent = f" ({abs(difference) / base_error * 100:.1f}%)" if base_error > 0 else ""
        if method == baseline:
            lines.append(f"The selected model did not beat {method_label(baseline)} on final MAE, so the existing release rule used that simple method instead.")
        else:
            lines.append(f"Compared with {method_label(baseline)} (MAE {base_error:.2f}), the selected model reduced average error by {difference:.2f} units{percent}.")
    elif selected == baseline:
        lines.append("A simple method already won validation, so the final table has only that method; there is no need to compare it with itself.")
    lines.append("The later check compares only the validation choice and strongest simple method; the other candidates were screened in validation. This release choice uses these results, so it does not provide another independent test afterward. Best in this run does not establish that the error is acceptable for your business.")
    return "\n\n".join(lines)


FORECAST_GUIDE = """This table contains the estimates that will be exported for the next seven days.

- area_id / category_id: stable identifiers; the added names tell you which place and category they represent.
- forecast_date: the date being estimated. horizon_day: days ahead of the last historical date (1 = next day, 7 = seven days ahead).
- predicted_units: expected recorded sale units for that date and combination. A decimal is an estimate, not a fraction of an item you must sell.
- lower_bound / upper_bound: an estimated low-to-high range would appear here if supported. Unavailable means this version does not calculate a range; it does not mean zero uncertainty.
- unit_policy: one recorded sellable item/package counts as one unit. Different sizes are not converted to kg or mL.

An area estimate combines participating stores; it is not an instruction for one store to buy that quantity."""


def forecast_explanation(forecast, method, labels=None, store=False):
    if forecast.empty:
        return "No forecast was produced for this scope. Review the withheld statuses; missing forecasts do not mean zero sales."
    keys = STORE_GROUP if store else GROUP
    count = len(forecast[keys].drop_duplicates())
    row = forecast.iloc[0]
    place = (labels or {}).get(row.area_id, str(row.area_id))
    who = f"store {row.store_id} in {place}" if store else f"participating stores in {place}"
    return (f"Using {method_label(method)}, produced {len(forecast):,} daily estimates across {count:,} "
            f"{'store/category' if store else 'area/category'} combinations for {forecast.forecast_date.min().date()} "
            f"through {forecast.forecast_date.max().date()}. The displayed rows are a preview. "
            f"For example, {category_label(row.category_id)} for {who} on {row.forecast_date.date()} "
            f"has an estimate of {row.predicted_units:.2f}, or approximately {row.predicted_units:.0f} recorded units "
            f"({int(row.horizon_day)} day(s) after the historical cutoff). This is an expectation, not a guarantee or a calculated error range.")


def feature_label(name):
    raw = name.split("__", 1)[-1]
    fixed = {"day_of_week": "Day of the week", "day_of_month": "Day of the month",
             "month": "Month", "is_weekend": "Whether the date is a weekend", "lag_1":"Sales yesterday"}
    if raw in fixed:
        return fixed[raw]
    for prefix, text in [("lag_", "Sales {n} day(s) earlier"), ("mean_", "Average daily sales over the previous {n} days"),
                         ("std_", "Variation in daily sales over the previous {n} days")]:
        if raw.startswith(prefix):
            return text.format(n=raw[len(prefix):])
    if raw.startswith("category_id_"):
        return "Category identity: " + category_label(raw[len("category_id_"):])
    if raw.startswith("area_id_"):
        area = raw[len("area_id_"):]
        return "Area identity: " + globals().get("area_labels", {}).get(area, area)
    if raw.startswith("store_id_"):
        return "Store identity: " + raw[len("store_id_"):]
    return "Identity indicator: " + raw


def feature_usage(estimator, method):
    if estimator is None or not hasattr(estimator.named_steps["model"], "feature_importances_"):
        text = ("The selected simple method has no tree feature-usage scores. Its forecast follows the matching weekdays described above."
                if method in BASELINES else "Feature-usage scores are unavailable because there is no fitted tree model for this scope.")
        return pd.DataFrame(), text
    model = estimator.named_steps["model"]
    names = estimator.named_steps["preprocess"].get_feature_names_out()
    scores = pd.Series(model.feature_importances_, index=names).nlargest(12)
    split = method == "lightgbm" and model.importance_type == "split"
    meaning = ("Number of tree splits" if split else "Relative model importance")
    table = pd.DataFrame({"Input used by the model": [feature_label(n) for n in scores.index], meaning: scores.values})
    if split:
        explanation = "These values count how many tree splits used each input. A score of 267 means 267 splits, not 267 sales or 267 percent."
    elif method == "random_forest":
        explanation = "These relative scores summarize reductions in training error when trees split on an input. They are not sales quantities."
    else:
        explanation = "These relative scores summarize the model's gain from splits on an input. They are not sales quantities."
    explanation += " Scores are specific to this model and should not be compared across model types. Related inputs can share credit. They describe overall model usage, not why a particular forecast rose or fell, and do not prove a cause."
    if not table.empty:
        explanation += f" The highest-ranked input here is: {table.iloc[0, 0]}."
    return table, explanation


def daily_chart_takeaway(actual, future, cutoff):
    observed = actual.loc[actual.units.notna()]
    lines = []
    if not observed.empty:
        lines.append(f"Displayed recorded daily sales range from {observed.units.min():.1f} to {observed.units.max():.1f} units.")
    if future is not None and not future.empty:
        complete = len(future) == 7 and future.units.notna().all()
        if complete:
            total = float(future.units.sum())
            lines.append(f"The next-seven-day estimate totals {total:.1f} recorded units.")
            previous = actual.set_index("date").reindex(pd.date_range(cutoff-pd.Timedelta(days=6), cutoff))
            if previous.units.notna().all() and previous.units.sum() > 0:
                change = (total / previous.units.sum() - 1) * 100
                lines.append(f"That is {abs(change):.1f}% {'higher' if change >= 0 else 'lower'} than the previous complete week's recorded sales.")
        else:
            lines.append("A full seven-day total is unavailable because some forecasts are missing; missing is not zero.")
    lines.append("Use recurring peaks or changes to decide what to investigate in sales and reporting records. Check promotions, stock availability, and store participation before attributing a cause; the chart alone cannot identify it.")
    return " ".join(lines)


def describe_figure(fig, guide, takeaway):
    # Keep Figure return values compatible with PNG export and existing callers.
    fig._bale_guide, fig._bale_takeaway = guide, takeaway
    return fig


def show_guided_figure(fig):
    from IPython.display import display
    import matplotlib.pyplot as plt
    print("How to read this chart:", fig._bale_guide)
    display(fig)
    print("What this shows:", fig._bale_takeaway)
    plt.close(fig)


def scope_summary(title, evaluation, method):
    lines = [f"### {title}", evaluation_explanation(evaluation, method)]
    ranked = evaluation.get("validation", []) if evaluation.get("mode") != "forecast_only" else []
    if ranked:
        lines.append("Validation ranking by average daily error: " + "; ".join(
            f"{i+1}. {method_label(r['method'])}: {r['MAE']:.2f} units" for i, r in enumerate(sorted(ranked, key=lambda r: r['MAE']))))
    return "\n\n".join(lines)


def run_summary(transactions, observations, statuses, forecast, audit, evaluation, method, cutoff,
                data_kind, estimator, store_evaluation=None, store_method=None, labels=None):
    lines = ["## What this run tells you", "",
             f"The goal is to estimate recorded sales for the next seven days, { (cutoff+pd.Timedelta(days=1)).date()} through {(cutoff+pd.Timedelta(days=7)).date()}.",
             f"This run uses {data_kind.upper()} data: {len(transactions):,} accepted sale lines, "
             f"{transactions.store_id.nunique():,} stores, {observations.area_id.nunique():,} areas and "
             f"{observations.category_id.nunique():,} categories. Historical dates span {observations.date.min().date()} through {cutoff.date()}.",
             cleaning_explanation(audit),
             f"{int(statuses.status.eq('eligible').sum()):,} area/category combinations qualify; "
             f"{int(statuses.status.ne('eligible').sum()):,} are withheld. Missing reporting is kept unknown, not replaced by zero.",
             scope_summary("Area/category result", evaluation, method)]
    if globals().get("SMOKE_RUN", False):
        lines.insert(2, f"Verification smoke run using {MODEL_TREES} trees per learned model. These reduced settings are for checking execution, not the normal model comparison.")
    if store_evaluation is not None:
        lines.append(scope_summary("Individual-store result (evaluated separately)", store_evaluation, store_method))
        lines.append("Store forecasts are modeled separately and may not add up to the area forecast.")
    evidence = evaluation.get("training_evaluation", evaluation)
    windows = evidence.get("windows", {})
    for stage, title in [("validation", "Earlier model-comparison dates"), ("final", "Later final-check dates")]:
        dates = windows.get(stage, [])
        if dates:
            lines.append(f"{title}: {(pd.Timestamp(min(dates))+pd.Timedelta(days=1)).date()} through {(pd.Timestamp(max(dates))+pd.Timedelta(days=7)).date()} ({len(dates)} seven-day tests).")
    lines += ["### What the inputs tell the model",
              "The learned models use past sales (1, 7, 14 and 28 days earlier), recent 7/14/28-day averages and variation, calendar dates, and area/category identities; the store model also uses store identity. They do not receive promotions, weather, prices, or inventory levels as predictive inputs.",
              feature_usage(estimator, method)[1],
              "### How the dataset affects this result",
              "The first 28 dates provide the history needed to build inputs. Longer history allows more test weeks; this run uses the evaluation dates stated above. More rows do not automatically mean better forecasts. Incomplete reporting can exclude combinations. Recorded sales can be limited by stockouts, so these estimates do not measure all unmet demand. A recorded package counts as one unit regardless of its contents; category totals can mix sizes."]
    if data_kind == "synthetic":
        lines.append("These sales are generated examples. Their model ranking and error scores test this workflow; they do not establish real business accuracy or real customer behavior.")
    lines += ["### Forecast highlights", forecast_explanation(forecast, method, labels)]
    changes = []
    for key, g in forecast.groupby(GROUP):
        past = observations.loc[observations.area_id.eq(key[0]) & observations.category_id.eq(key[1]) &
                                observations.date.between(cutoff-pd.Timedelta(days=6), cutoff)]
        total = float(g.predicted_units.sum())
        if len(g) == 7 and len(past) == 7 and past.units_sold.notna().all() and past.units_sold.sum() > 0:
            changes.append((abs(total/past.units_sold.sum()-1), (labels or {}).get(key[0], key[0]), key[1], total,
                            (total/past.units_sold.sum()-1)*100))
    for _, place, category, total, change in sorted(changes, key=lambda r: r[0], reverse=True)[:3]:
        lines.append(f"- {place} / {category_label(category)}: {total:.1f} units expected over seven days, {change:+.1f}% versus the previous complete week. This is a forecasted change, not an observed future outcome or an explanation of its cause.")
    if not changes:
        lines.append("No complete positive previous-week comparison is available.")
    lines += ["### What to do with the result",
              "Review the forecast alongside your own stock and sales records. Investigate large changes and historical misses. No prediction range is calculated, and no business acceptance threshold has been declared. Use new real outcomes to check accuracy before relying on the result for decisions."]
    return "\n\n".join(lines) + "\n"


In [ ]:
# The city demo is self-contained: missing files are generated locally without downloads.
# Explicit legacy GENERATE_DEMO=True takes precedence; DEMO_PRESET=None uses real inputs.
if DEMO_PRESET not in {None, "panabo_50k"}:
    raise ValueError("DEMO_PRESET must be None or 'panabo_50k'.")
if DEMO_PRESET == "panabo_50k" and not GENERATE_DEMO:
    preset = panabo_preset(DEMO_DIRECTORY, PANABO_REFERENCE)
    CSV_PATH, COVERAGE_PATH = preset["csv_path"], preset["coverage_path"]
    COORDINATES_PATH, AREA_REGISTRY = preset["coordinates_path"], preset["area_registry"]
    FORECAST_CUTOFF = FORECAST_CUTOFF or preset["cutoff"]
    AS_OF = AS_OF or preset["as_of"]
    DATA_KIND = "synthetic"
    CATEGORY_LABELS = {c[0]:c[1] for c in PANABO_CATEGORIES}
    print("PANABO SYNTHETIC DEMO |", preset["manifest"]["rows"], "sale lines | 40 barangays | 80 stores | 4 categories")

if MODE not in {"train", "forecast_only"} or FORECAST_DAYS != 7:
    raise ValueError("Use mode train or forecast_only and a seven-day horizon.")
as_of = parse_timestamp(AS_OF if AS_OF else datetime.now(timezone.utc))
if pd.isna(as_of):
    raise ValueError("AS_OF must be an offset timestamp.")
last_complete_day = as_of.tz_convert("Asia/Manila").tz_localize(None).normalize() - pd.Timedelta(days=1)
cutoff = pd.Timestamp(FORECAST_CUTOFF).normalize() if FORECAST_CUTOFF else last_complete_day
if cutoff > last_complete_day:
    raise ValueError("The forecast cutoff cannot include an incomplete or future Manila date.")
if GENERATE_DEMO:
    CSV_PATH, COVERAGE_PATH = generate_demo("demo_data", cutoff - pd.Timedelta(days=1), seed=RANDOM_SEED)
    DATA_KIND = "synthetic"
    cutoff -= pd.Timedelta(days=1)  # permit all next-midnight demo reporting to be available
if DATA_KIND not in {"real", "synthetic"}:
    raise ValueError("Declare DATA_KIND as real or synthetic.")

import joblib
saved_bundle = None
config = current_config()
if MODE == "forecast_only":
    if SAVED_MODEL_PATH is None:
        raise ValueError("Set SAVED_MODEL_PATH to a trusted method.joblib; never load untrusted pickle files.")
    saved_bundle = joblib.load(choose_input(SAVED_MODEL_PATH))
    if saved_bundle["feature_version"] != FEATURE_VERSION or saved_bundle["data_kind"] != DATA_KIND:
        raise ValueError("Incompatible feature version or data kind.")
    if cutoff < pd.Timestamp(saved_bundle["model_training_end"]):
        raise ValueError("Forecast cutoff predates the model's training data.")
    config = saved_bundle["cleaning_config"]
    print("Using saved canonicalization and cleaning mappings.")
CSV_PATH = choose_input(CSV_PATH)
raw = pd.read_csv(CSV_PATH, dtype="string", keep_default_na=False)
transactions, quarantine, duplicates, blocked_groups, cleaning_audit = clean_transactions(raw, config)
print("Why cleaning matters: duplicated, invalid, or incomplete sale lines can distort the sales being predicted.")
print(cleaning_explanation(cleaning_audit))
area_labels = {r.area_id:f"{r.barangay.title()}, {r.city.title()}" for r in transactions[["area_id", "barangay", "city"]].drop_duplicates().itertuples()}
if not quarantine.empty:
    print("Quarantined rows (source CSV row number and reasons):")
    display(quarantine[["source_row", "reason"]].head(30))
display(transactions[["city", "barangay", "area_id"]].drop_duplicates())
print("Review canonical labels for remaining misspellings; add aliases and rerun before trusting results.")
print("Check the category names and package types below. Count means sale lines; sum means recorded units sold, not weight or volume.")
display(readable_table(transactions.groupby(["category_id", "unit_label"]).quantity.agg(["count", "sum"]).reset_index()))
print("Cleaning summary: only accepted completed sales enter the forecast. Repair rejected identities or quantities before interpreting affected combinations.")
if cleaning_audit["global_block"]:
    raise ValueError("Unlocatable quarantined records could affect any group. Repair identities and rerun. Audit is displayed above.")


## Reporting evidence and daily observations

Optional `coverage.csv`: one row per store/category/Manila date with either canonical
`area_id` or `city,barangay`, plus `participation_status` (`active`, `not_started`, `ended`),
`reporting_status` (`complete`, `partial`, `unknown`), `operating_status`
(`open`, `closed`, `unknown`), `availability_status` (`available`, `stockout`, `unknown`),
offset `known_at`, and `coverage_source`.

For each issuance, its participant cohort is held fixed across its historical grid.
Evaluation outcomes must have the same cohort. Missing or partial statements remain unknown. A confirmed closed
day can be zero sales, but unknown or absent reporting cannot. This conservative policy
can reject new/changing cohorts. Future participation changes require a fresh review.


In [ ]:
def prepare_coverage(frame, config):
    cov = frame.copy()
    required = {"store_id", "category_id", "date", "participation_status", "reporting_status",
                "operating_status", "availability_status", "known_at", "coverage_source"}
    if not required.issubset(cov):
        raise ValueError("Coverage missing: " + ", ".join(sorted(required-set(cov.columns))))
    if {"city", "barangay"}.issubset(cov):
        cov["area_id"] = [location[2] for location in canonical_locations(cov.city, cov.barangay, config)]
    if "area_id" not in cov:
        raise ValueError("Coverage needs area_id or city and barangay.")
    for col in ["store_id", "category_id", "area_id", "coverage_source"]:
        cov[col] = cov[col].astype("string").str.strip()
        if cov[col].isna().any() or cov[col].eq("").any():
            raise ValueError("Empty coverage identity/source: " + col)
    cov["date"] = pd.to_datetime(cov.date, format="%Y-%m-%d", errors="coerce")
    cov["known_at"] = pd.to_datetime(cov.known_at.map(lambda v: parse_timestamp(v, config["source_timezone"])), utc=True)
    if cov.date.isna().any() or cov.known_at.isna().any():
        raise ValueError("Coverage has invalid dates or known_at timestamps.")
    enums = {"participation_status": {"active", "not_started", "ended"},
             "reporting_status": {"complete", "partial", "unknown"},
             "operating_status": {"open", "closed", "unknown"},
             "availability_status": {"available", "stockout", "unknown"}}
    for col, values in enums.items():
        cov[col] = cov[col].map(normalized)
        if not cov[col].isin(values).all():
            raise ValueError("Unknown coverage enum: " + col)
    cov = cov.drop_duplicates()
    if cov.duplicated(["store_id", "category_id", "date"]).any():
        raise ValueError("Conflicting coverage rows; provide one reconciled snapshot per store/category/date.")
    return cov

def assumed_coverage(config):
    if not ASSUME_COMPLETE_REPORTING or not REPORTING_START or not REPORTING_END or not DECLARED_COHORT:
        raise ValueError("No coverage file: explicitly set ASSUME_COMPLETE_REPORTING, REPORTING_START, REPORTING_END, and DECLARED_COHORT.")
    dates = pd.date_range(REPORTING_START, REPORTING_END)
    if not len(dates):
        raise ValueError("Invalid declared reporting range.")
    rows = []
    for member in DECLARED_COHORT:
        for date in dates:
            rows.append({**member, "date": str(date.date()), "participation_status":"active",
                "reporting_status":"complete", "operating_status":"unknown", "availability_status":"unknown",
                # Assumed historical availability, not measured system knowledge.
                "known_at":date.tz_localize("Asia/Manila").isoformat(),
                "coverage_source":"explicit_user_completeness_assumption"})
    return prepare_coverage(pd.DataFrame(rows), config)

def issuance_time(date):
    # Fixed daily issuance: 00:15 Manila, allowing the declared reporting close.
    return (pd.Timestamp(date) + pd.Timedelta(days=1, minutes=15)).tz_localize("Asia/Manila").tz_convert("UTC")

def daily_panel(tx, cov, end_date, knowledge_time, blocked, group_columns=None):
    group_columns = list(group_columns or GROUP)
    end_date = pd.Timestamp(end_date)
    c = cov.loc[(cov.date <= end_date) & (cov.known_at <= knowledge_time)].copy()
    t = tx.loc[(tx.date <= end_date) & (tx.known_at <= knowledge_time)].copy()
    # Cohort is determined only by information available at this cutoff.
    latest = c.sort_values("date").groupby(["store_id", "category_id"], as_index=False).tail(1)
    cohort = latest.loc[latest.participation_status.eq("active"), list(dict.fromkeys(["store_id", *group_columns]))].drop_duplicates()
    # Sales outside declared participants/evidence are a reconciliation failure.
    tx_check = t.merge(c[list(dict.fromkeys(["store_id", *group_columns, "date", "participation_status", "operating_status"]))],
                       on=list(dict.fromkeys(["store_id", *group_columns, "date"])), how="left")
    bad = tx_check.participation_status.isna() | tx_check.participation_status.ne("active") | tx_check.operating_status.eq("closed")
    bad_keys = set(map(tuple, tx_check.loc[bad, group_columns].drop_duplicates().to_numpy()))
    known_blocked = blocked
    if "known_at" in blocked:
        known_blocked = blocked.loc[(blocked.known_at.isna() | (blocked.known_at <= knowledge_time)) &
            (blocked.transaction_datetime.isna() | (blocked.transaction_datetime < issuance_time(end_date)))]
    if "store_id" in group_columns:
        if "store_id" not in known_blocked:
            known_blocked = known_blocked.assign(store_id="")
        broad = known_blocked.loc[known_blocked.store_id.fillna("").eq("")].drop(columns="store_id")
        expanded = broad.merge(cohort, on=["area_id", "category_id"], how="inner")
        known_blocked = pd.concat([known_blocked, expanded], ignore_index=True)
    bad_keys |= set(map(tuple, known_blocked[group_columns].to_numpy()))
    rows = []
    coverage_groups = dict(tuple(c.groupby(group_columns, sort=False)))
    transaction_groups = dict(tuple(t.groupby(group_columns, sort=False)))
    for key, members in cohort.groupby(group_columns, sort=True):
        group_cov = coverage_groups[key]
        dates = pd.date_range(group_cov.date.min(), end_date)
        expected = pd.MultiIndex.from_product([members.store_id.unique(), dates], names=["store_id", "date"]).to_frame(index=False)
        observed = expected.merge(group_cov, on=["store_id", "date"], how="left", validate="one_to_one")
        observed["complete"] = observed.reporting_status.eq("complete") & observed.participation_status.eq("active")
        complete = observed.groupby("date").complete.all().reindex(dates, fill_value=False)
        sales = transaction_groups.get(key, t.iloc[:0])
        sales = sales.loc[sales.store_id.isin(members.store_id)]
        totals = sales.groupby("date").quantity.sum().astype(float).reindex(dates, fill_value=0.)
        if key in bad_keys:
            complete[:] = False
        cohort_id = hashlib.sha256(json.dumps(sorted(members.store_id.tolist())).encode()).hexdigest()[:16]
        for date in dates:
            rows.append(dict(zip(group_columns, key)) | dict(date=date,
                units_sold=float(totals.loc[date]) if complete.loc[date] else np.nan,
                observation_status="complete" if complete.loc[date] else "incomplete_reporting",
                store_count=len(members), cohort_id=cohort_id))
    # Keep uncovered sale groups visible as withheld rather than silently dropping them.
    represented = set(map(tuple, cohort[group_columns].drop_duplicates().to_numpy()))
    for key, sales in transaction_groups.items():
        if key not in represented:
            for date in pd.date_range(sales.date.min(), end_date):
                rows.append(dict(zip(group_columns, key)) | dict(date=date,
                    units_sold=np.nan, observation_status="incomplete_reporting",
                    store_count=int(sales.store_id.nunique()), cohort_id="unreconciled"))
    return pd.DataFrame(rows, columns=[*group_columns, "date", "units_sold", "observation_status", "store_count", "cohort_id"])

def eligibility(panel, end_date, group_columns=None):
    group_columns = list(group_columns or GROUP)
    rows = []
    for key, g in panel.groupby(group_columns, sort=True):
        recent = g.set_index("date").reindex(pd.date_range(pd.Timestamp(end_date)-pd.Timedelta(days=27), end_date))
        status = "eligible"
        if g.units_sold.notna().sum() < 56:
            status = "insufficient_history"
        elif recent.units_sold.isna().any():
            status = "incomplete_reporting"
        elif recent.units_sold.gt(0).sum() < 7:
            status = "no_sales" if recent.units_sold.fillna(0).eq(0).all() else "sparse_sales"
        rows.append(dict(zip(group_columns, key)) | dict(status=status,
                         complete_dates=int(g.units_sold.notna().sum()), store_count=int(g.store_count.iloc[-1]),
                         cohort_id=g.cohort_id.iloc[-1]))
    return pd.DataFrame(rows, columns=[*group_columns, "status", "complete_dates", "store_count", "cohort_id"])

def eligible_panel(panel, statuses, group_columns=None):
    group_columns = list(group_columns or GROUP)
    return panel.merge(statuses.loc[statuses.status.eq("eligible"), group_columns], on=group_columns)


In [ ]:
coverage_assumed = COVERAGE_PATH is None
if coverage_assumed:
    coverage = assumed_coverage(config)
else:
    COVERAGE_PATH = choose_input(COVERAGE_PATH)
    coverage = prepare_coverage(pd.read_csv(COVERAGE_PATH, dtype="string", keep_default_na=False), config)

# At historical issuance, never read records received after that issuance.
# Current final forecasts may use all data actually available by this run's AS_OF.
if transactions.category_id.eq(ALL_CATEGORIES).any() or coverage.category_id.eq(ALL_CATEGORIES).any():
    raise ValueError("Source category ID is reserved for the All categories selection.")
panel = daily_panel(transactions, coverage, cutoff, as_of, blocked_groups)
coverage_summary = eligibility(panel, cutoff)
print("Reporting evidence:", "ASSUMED COMPLETE" if coverage_assumed else "supplied coverage statements")
print("A combination is one area and category. Eligible means it has enough complete recent reporting and sales activity to forecast. Other statuses explain why it is withheld; missing reporting is not zero sales.")
display(readable_table(coverage_summary, area_labels))
print(f"Reporting summary: {int(coverage_summary.status.eq('eligible').sum())} area/category combinations qualify; {int(coverage_summary.status.ne('eligible').sum())} are withheld. Supplied or assumed reporting completeness is not independently audited here.")
if panel.empty:
    raise ValueError("No active observed groups at cutoff. Check coverage, known_at, and dates.")
print("Calendar span:", panel.date.min().date(), "through", cutoff.date())
print("180 days is a starting recommendation, not an accuracy guarantee. 90 days uses fewer test windows.")


## What information can a model learn from?

For each date, the learned models receive earlier sales, recent sales averages and variation, calendar information, and area/category identities. The store model also receives store identity. A feature is simply one such input, not a proven reason for demand.

The model predicts the next day, then uses that estimate when calculating later days in the same seven-day forecast. Errors can therefore carry forward. Nothing from the actual future week is used as an input at issuance.


In [ ]:
NUMERIC_FEATURES = ["day_of_week", "day_of_month", "month", "is_weekend",
                    *[f"lag_{n}" for n in LAGS],
                    *[f"mean_{n}" for n in WINDOWS], *[f"std_{n}" for n in WINDOWS]]
FEATURE_COLUMNS = GROUP + NUMERIC_FEATURES

def feature_row(key, date, past, group_columns=None):
    group_columns = list(group_columns or GROUP)
    values = np.asarray(past, dtype=float)
    if len(values) < 28 or not np.isfinite(values[-28:]).all():
        return None
    row = dict(zip(group_columns, key))
    row.update(day_of_week=date.dayofweek, day_of_month=date.day, month=date.month,
               is_weekend=int(date.dayofweek >= 5))
    row.update({f"lag_{n}":float(values[-n]) for n in LAGS})
    for n in WINDOWS:
        row[f"mean_{n}"] = float(values[-n:].mean())
        row[f"std_{n}"] = float(values[-n:].std(ddof=1))
    return row

def training_examples(panel, group_columns=None):
    group_columns = list(group_columns or GROUP)
    examples = []
    for key, g in panel.groupby(group_columns, sort=True):
        g = g.sort_values("date")
        past = []
        for row in g.itertuples():
            features = feature_row(key, row.date, past, group_columns)
            if features is not None and np.isfinite(row.units_sold):
                examples.append({**features, "date":row.date, "units_sold":row.units_sold})
            past.append(row.units_sold)
    return pd.DataFrame(examples, columns=[*group_columns, *NUMERIC_FEATURES, "date", "units_sold"])

def build_estimator(method, group_columns=None):
    group_columns = list(group_columns or GROUP)
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder
    from xgboost import XGBRegressor
    from lightgbm import LGBMRegressor
    preprocessing = ColumnTransformer([
        ("identities", OneHotEncoder(handle_unknown="ignore", sparse_output=False), group_columns),
        ("numeric", "passthrough", NUMERIC_FEATURES)]).set_output(transform="pandas")
    estimators = {
        "random_forest":RandomForestRegressor(n_estimators=MODEL_TREES, max_depth=16,
            min_samples_leaf=3, n_jobs=2, random_state=RANDOM_SEED),
        "xgboost":XGBRegressor(n_estimators=MODEL_TREES, max_depth=5, learning_rate=.05,
            objective="reg:squarederror", tree_method="hist", n_jobs=2, random_state=RANDOM_SEED),
        "lightgbm":LGBMRegressor(n_estimators=MODEL_TREES, num_leaves=15, learning_rate=.05,
            verbosity=-1, n_jobs=2, random_state=RANDOM_SEED),
    }
    return Pipeline([("preprocess", preprocessing), ("model", estimators[method])])

BASELINES = ["last_week", "weekday_average"]
METHODS = BASELINES + ["random_forest", "xgboost", "lightgbm"]

def fit_method(method, history, group_columns=None):
    group_columns = list(group_columns or GROUP)
    if method in BASELINES:
        return None
    examples = training_examples(history, group_columns)
    if examples.empty:
        raise ValueError("No training examples remain after lag warmup and reporting checks.")
    estimator = build_estimator(method, group_columns)
    estimator.fit(examples[[*group_columns, *NUMERIC_FEATURES]], examples.units_sold)
    return estimator

def predict_seven(method, estimator, history, end_date, group_columns=None):
    group_columns = list(group_columns or GROUP)
    if history.empty:
        return pd.DataFrame(columns=[*group_columns, "forecast_date", "horizon_day", "predicted_units"])
    states = {key:g.sort_values("date").units_sold.tolist() for key, g in history.groupby(group_columns, sort=True)}
    rows = []
    for horizon in range(1, 8):
        date = pd.Timestamp(end_date) + pd.Timedelta(days=horizon)
        keys, features = [], []
        for key, values in states.items():
            row = feature_row(key, date, values, group_columns)
            if row is None:
                raise ValueError("Forecast requires complete recent 28-day history.")
            keys.append(key)
            features.append(row)
        if method == "last_week":
            predicted = [states[key][-7] for key in keys]
        elif method == "weekday_average":
            predicted = [np.mean([states[key][-n] for n in [7, 14, 21, 28]]) for key in keys]
        else:
            predicted = estimator.predict(pd.DataFrame(features)[[*group_columns, *NUMERIC_FEATURES]])
        for key, value in zip(keys, predicted):
            if not np.isfinite(value):
                raise ValueError("Nonfinite model prediction.")
            value = max(0., float(value))
            states[key].append(value)
            rows.append(dict(zip(group_columns, key)) | dict(forecast_date=date, horizon_day=horizon, predicted_units=value))
    return pd.DataFrame(rows)

def metrics(frame):
    if frame.empty:
        return {"MAE":None, "RMSE":None, "bias":None, "WAPE_pct":None, "rows":0}
    error = frame.predicted_units.to_numpy() - frame.actual_units.to_numpy()
    denominator = float(frame.actual_units.abs().sum())
    return {"MAE":float(np.abs(error).mean()), "RMSE":float(np.sqrt(np.mean(error**2))),
            "bias":float(error.mean()), "WAPE_pct":float(np.abs(error).sum()/denominator*100) if denominator else None,
            "rows":len(frame)}

def evaluation_windows(start, end):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    span = (end-start).days + 1
    if span < 84:
        raise ValueError("Evaluated training needs at least 84 calendar dates and eligible history per group.")
    if span >= 180:
        validation = [end-pd.Timedelta(days=n) for n in [84, 77, 70, 63]]
        calibration = [end-pd.Timedelta(days=n) for n in [56, 49, 42, 35]]
        final = [end-pd.Timedelta(days=n) for n in [28, 21, 14, 7]]
        protocol = "four_validation_four_reserved_calibration_four_final_weeks"
    else:
        validation = [end-pd.Timedelta(days=n) for n in [28, 21]]
        calibration = []
        final = [end-pd.Timedelta(days=n) for n in [14, 7]]
        protocol = "short_history_two_validation_two_final_weeks"
    return {"validation":validation, "reserved_calibration":calibration, "final":final, "protocol":protocol}

def backtest(tx, cov, blocked, cutoffs, methods, group_columns=None):
    group_columns = list(group_columns or GROUP)
    predictions, refusals = [], []
    for end in cutoffs:
        history = daily_panel(tx, cov, end, issuance_time(end), blocked, group_columns)
        statuses = eligibility(history, end, group_columns)
        for row in statuses.to_dict("records"):
            if row["status"] != "eligible":
                refusals.append({**row, "cutoff":str(end.date()), "reason":row["status"]})
        usable = eligible_panel(history, statuses, group_columns)
        if usable.empty:
            continue
        scope = "store/category" if "store_id" in group_columns else "area/category"
        print(f"Using history through {end.date()} to predict {(end+pd.Timedelta(days=1)).date()} to {(end+pd.Timedelta(days=7)).date()}: {len(usable[group_columns].drop_duplicates())} eligible {scope} combinations")
        # Outcomes become available by the later evaluation snapshot; not model inputs.
        truth = daily_panel(tx, cov, end+pd.Timedelta(days=7), as_of, blocked, group_columns)
        for method in methods:
            print("  Checking:", method_label(method), flush=True)
            estimator = fit_method(method, usable, group_columns)
            forecast = predict_seven(method, estimator, usable, end, group_columns)
            joined = forecast.merge(truth[[*group_columns, "date", "units_sold", "cohort_id"]],
                left_on=[*group_columns, "forecast_date"], right_on=[*group_columns, "date"], how="left")
            cohort_at_issue = statuses[[*group_columns, "cohort_id"]].rename(columns={"cohort_id":"issued_cohort_id"})
            joined = joined.merge(cohort_at_issue, on=group_columns)
            scoreable = joined.units_sold.notna() & joined.cohort_id.eq(joined.issued_cohort_id)
            for key in joined.loc[~scoreable, group_columns].drop_duplicates().itertuples(index=False, name=None):
                refusals.append(dict(zip(group_columns, key)) | {"cutoff":str(end.date()), "method":method,
                    "reason":"unavailable_outcome_or_changed_cohort"})
            joined = joined.loc[scoreable].rename(columns={"units_sold":"actual_units"})
            joined["method"], joined["cutoff"] = method, end
            predictions.append(joined[[*group_columns, "cutoff", "forecast_date", "horizon_day", "method", "predicted_units", "actual_units"]])
    columns = [*group_columns, "cutoff", "forecast_date", "horizon_day", "method", "predicted_units", "actual_units"]
    return pd.concat(predictions, ignore_index=True) if predictions else pd.DataFrame(columns=columns), pd.DataFrame(refusals)

def compare_methods(predictions):
    return pd.DataFrame([{"method":method, **metrics(g)} for method, g in predictions.groupby("method")])

class EvaluationEvidenceError(ValueError):
    """This scope lacks comparable evidence; withhold its evaluated forecasts."""


def evaluate_scope(panel, tx, cov, blocked, cutoff, group_columns, limitations):
    evaluation = {"mode":"train", "intervals":"unavailable", "limitations":list(limitations)}
    try:
        windows = evaluation_windows(panel.date.min(), cutoff)
    except ValueError as error:
        raise EvaluationEvidenceError(str(error)) from error
    evaluation["windows"] = {k:[str(d.date()) for d in v] if isinstance(v, list) else v for k, v in windows.items()}
    scope_label = "individual stores and categories" if "store_id" in group_columns else "areas and categories"
    print(f"Evaluating {scope_label} using {DATA_KIND.upper()} data.")
    print(f"Stage 1: compare all five methods across {len(windows['validation'])} earlier weeks (validation). "
          f"Stage 2: check the chosen method against the strongest simple method across {len(windows['final'])} later weeks (final check).")
    print(f"{len(windows['reserved_calibration'])} intervening weeks are reserved for future uncertainty work; no prediction ranges are calculated in this version.")
    print(METRIC_GUIDE)
    started = time.perf_counter()
    validation_predictions, val_refusals = backtest(tx, cov, blocked, windows["validation"], METHODS, group_columns)
    comparison = compare_methods(validation_predictions)
    if comparison.empty or set(comparison.method) != set(METHODS):
        raise EvaluationEvidenceError("Insufficient shared validation evidence for all candidates. Inspect history/coverage.")
    # Same cutoffs/groups/rows for every method; fail instead of scoring unlike samples.
    row_keys = group_columns + ["cutoff", "forecast_date"]
    counts = validation_predictions.groupby(row_keys).method.nunique()
    if not counts.eq(len(METHODS)).all():
        raise EvaluationEvidenceError("Candidate evaluation rows are not identical.")
    comparison["tie_priority"] = comparison.method.map({m:i for i, m in enumerate(METHODS)})
    comparison = comparison.sort_values(["MAE", "tie_priority"])
    selected_method = comparison.iloc[0].method
    best_baseline = comparison.loc[comparison.method.isin(BASELINES)].iloc[0].method
    print("Stage 1 results: all five methods, ranked by MAE.")
    display(readable_table(comparison.drop(columns="tie_priority")))
    print(validation_explanation(comparison))
    print("Strongest simple comparison method:", method_label(best_baseline))
    print("Stage 2 includes only the validation choice and this simple method. The other models were screened in Stage 1; a duplicate choice appears once.")
    finalists = list(dict.fromkeys([selected_method, best_baseline]))
    final_predictions, final_refusals = backtest(tx, cov, blocked, windows["final"], finalists, group_columns)
    final_comparison = compare_methods(final_predictions)
    if set(final_comparison.method) != set(finalists) or final_predictions.empty:
        raise EvaluationEvidenceError("No comparable final-test evidence; no evaluated release will be exported.")
    if not final_predictions.groupby(row_keys).method.nunique().eq(len(finalists)).all():
        raise EvaluationEvidenceError("Final evaluation rows are not identical.")
    final_mae = final_comparison.set_index("method").MAE
    published_method = selected_method
    if selected_method not in BASELINES and final_mae[selected_method] >= final_mae[best_baseline]:
        published_method = best_baseline
    evaluation.update(validation_selected_method=selected_method, strongest_validation_baseline=best_baseline,
        published_method=published_method, validation=comparison.drop(columns="tie_priority").to_dict("records"),
        final=final_comparison.to_dict("records"), duration_seconds=time.perf_counter()-started)
    evaluation_refusals = pd.concat([val_refusals.assign(stage="validation"), final_refusals.assign(stage="final")], ignore_index=True)
    print("Stage 2 results: later-week check of the selected finalists.")
    display(readable_table(final_comparison))
    print(evaluation_explanation(evaluation, published_method))
    return {"evaluation":evaluation, "comparison":comparison, "validation_predictions":validation_predictions,
            "final_predictions":final_predictions, "refusals":evaluation_refusals,
            "published_method":published_method, "windows":windows}

def candidate_forecasts(history, cutoff, group_columns, published_method, published_estimator):
    if history.empty:
        return pd.DataFrame(columns=[*group_columns, "forecast_date", "horizon_day", "predicted_units", "method"])
    frames = []
    for method in TREE_METHODS:
        fitted = published_estimator if method == published_method else fit_method(method, history, group_columns)
        frame = predict_seven(method, fitted, history, cutoff, group_columns)
        frame["method"] = method
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)

def restrict_saved_groups(statuses, saved, group_columns):
    result = statuses.copy()
    supported = set(map(tuple, saved["supported_groups"]))
    for index, row in result.iterrows():
        key = tuple(row[k] for k in group_columns)
        if row.status == "eligible" and (key not in supported or saved["cohorts"].get(json.dumps(key)) != row.cohort_id):
            result.loc[index, "status"] = "new_group_or_changed_cohort_requires_retraining"
    return result


## Which forecasting method works best here?

We replay past weeks: use only sales and reporting information available at that
time, predict the following seven days, then compare those estimates with recorded sales.
This checks performance on dates the model had not yet seen.

**Stage 1 (validation)** compares Random Forest, XGBoost, LightGBM, and two simple
alternatives: repeat last week, or average the previous four matching weekdays.
**MAE chooses the method** because our goal is to reduce the average number of units
missed per daily category estimate. RMSE also checks errors while giving bigger misses
more weight; it can favor a different model without making the MAE decision invalid.

**Stage 2 (final check)** compares the validation choice with the strongest simple
alternative on later weeks. If the learned model does not improve MAE, we use the
simple method. Only these finalists appear in the second table. This choice uses the
final-check scores, so those scores are not a new independent test after that decision.

With 180 or more historical dates, there are four validation weeks, four intervening
weeks reserved for future uncertainty calibration, and four final-check weeks. Reserved
weeks do not choose the method; this version calculates no prediction ranges. With
84-179 dates, we use two validation and two final-check weeks. Earlier dates train
each model; fewer than 84 dates cannot support this evaluated workflow.


In [ ]:
evaluation = {"mode":MODE, "intervals":"unavailable", "limitations":[
    "Retrospective current-state evaluation: final status and current mappings cannot reconstruct historical corrections.",
    "Recorded sales may be constrained by stockouts; future store participation is assumed unchanged.",
    "Coverage completeness is assumed by the operator." if coverage_assumed else "Coverage statements are supplied, not independently audited."]}
comparison = pd.DataFrame()
validation_predictions = final_predictions = pd.DataFrame()
evaluation_refusals = pd.DataFrame()
windows = None
if MODE == "train":
    area_result = evaluate_scope(panel, transactions, coverage, blocked_groups, cutoff, GROUP, evaluation["limitations"])
    evaluation = area_result["evaluation"]
    comparison = area_result["comparison"]
    validation_predictions = area_result["validation_predictions"]
    final_predictions = area_result["final_predictions"]
    evaluation_refusals = area_result["refusals"]
    published_method = area_result["published_method"]
    windows = area_result["windows"]
else:
    published_method = saved_bundle["method"]
    evaluation["training_evaluation"] = saved_bundle["evaluation"]
    evaluation["model_training_end"] = saved_bundle["model_training_end"]
    evaluation["message"] = "No retraining or new accuracy measurement; original evaluation belongs to the saved model."


In [ ]:
statuses = coverage_summary.copy()
history = eligible_panel(panel, statuses)
if MODE == "forecast_only":
    supported = set(map(tuple, saved_bundle["supported_groups"]))
    for index, row in statuses.iterrows():
        key = tuple(row[k] for k in GROUP)
        if row.status == "eligible" and (key not in supported or saved_bundle["cohorts"].get(json.dumps(key)) != row.cohort_id):
            statuses.loc[index, "status"] = "new_group_or_changed_cohort_requires_retraining"
    history = eligible_panel(panel, statuses)
    estimator = saved_bundle["estimator"]
else:
    estimator = fit_method(published_method, history) if not history.empty else None
forecast = predict_seven(published_method, estimator, history, cutoff)
forecast["lower_bound"] = None
forecast["upper_bound"] = None
forecast["unit_policy"] = "recorded_sale_units"
print(FORECAST_GUIDE)
display(readable_table(forecast.head(14), area_labels))
print(forecast_explanation(forecast, published_method, area_labels))
print("These statuses explain any combination without a forecast:")
display(readable_table(statuses, area_labels))


## Individual-store forecasts and map data
The store pipeline is evaluated independently from barangay totals. Its predictions need
not sum to the barangay prediction. Store outputs are private operator handoff data.
Real map placement requires a coordinate sidecar; synthetic points are illustrative.


In [ ]:
def prepare_coordinates(frame):
    required = ["area_id", "latitude", "longitude", "coordinate_source"]
    if not set(required).issubset(frame.columns):
        raise ValueError("Coordinate sidecar needs: " + ", ".join(required))
    out = frame[required].copy()
    for name in ["area_id", "coordinate_source"]:
        out[name] = out[name].astype("string").str.strip()
        if out[name].isna().any() or out[name].eq("").any():
            raise ValueError("Coordinate identity and source cannot be empty.")
    for name, limit in [("latitude", 90), ("longitude", 180)]:
        original = out[name].replace("", pd.NA)
        values = pd.to_numeric(original, errors="coerce")
        if (original.notna() & (values.isna() | ~np.isfinite(values) | values.abs().gt(limit))).any():
            raise ValueError("Invalid " + name)
        out[name] = values.astype(float)
    if (out.latitude.isna() != out.longitude.isna()).any():
        raise ValueError("Supply both coordinates or neither.")
    out = out.drop_duplicates()
    if out.area_id.duplicated().any():
        raise ValueError("Conflicting coordinate rows for one area.")
    return out

def demo_coordinates(area_ids):
    # Intentionally illustrative points, NOT authoritative barangay centroids.
    rows = [{"area_id":area, "latitude":7.29 + i*.012, "longitude":125.67 + i*.009,
             "coordinate_source":"synthetic_illustrative_not_barangay_centroid"}
            for i, area in enumerate(sorted(area_ids))]
    return prepare_coordinates(pd.DataFrame(rows, columns=["area_id", "latitude", "longitude", "coordinate_source"]))

def selected_daily(observations, forecasts, area_id, category_id=ALL_CATEGORIES, store_id=None):
    """Strict daily totals: absent/unknown categories never become zero or partial sums."""
    mask = observations.area_id.eq(area_id)
    if store_id is not None:
        mask &= observations.store_id.eq(store_id)
    past = observations.loc[mask].copy()
    future = forecasts.loc[forecasts.area_id.eq(area_id)].copy()
    if store_id is not None:
        future = future.loc[future.store_id.eq(store_id)]
    categories = sorted(past.category_id.unique()) if category_id == ALL_CATEGORIES else [category_id]
    past = past.loc[past.category_id.isin(categories)]
    future = future.loc[future.category_id.isin(categories)]
    def total(frame, date_col, value_col):
        rows = []
        for date, g in frame.groupby(date_col, sort=True):
            complete = (len(g) == len(categories) and g.category_id.nunique() == len(categories)
                        and g[value_col].notna().all() and np.isfinite(g[value_col]).all())
            rows.append({"date":date, "units":float(g[value_col].sum()) if complete else np.nan})
        return pd.DataFrame(rows, columns=["date", "units"])
    return total(past, "date", "units_sold"), total(future, "forecast_date", "predicted_units")

def map_summaries(observations, forecasts, cutoff):
    columns = ["area_id", "category_id", "view", "lookback_days", "start_date", "end_date",
               "total_units", "coverage_status", "observed_days", "expected_days", "category_count"]
    rows = []
    for area, g in observations.groupby("area_id", sort=True):
        for category in [ALL_CATEGORIES, *sorted(g.category_id.unique())]:
            past, future = selected_daily(observations, forecasts, area, category)
            count = g.category_id.nunique() if category == ALL_CATEGORIES else 1
            for view, days in [("historical", 90), ("historical", 180), ("forecast", 7)]:
                start = cutoff-pd.Timedelta(days=days-1) if view == "historical" else cutoff+pd.Timedelta(days=1)
                end = cutoff if view == "historical" else cutoff+pd.Timedelta(days=7)
                series = past if view == "historical" else future
                series = series.set_index("date").reindex(pd.date_range(start, end))
                observed = int(series.units.notna().sum())
                complete = observed == days
                rows.append(dict(area_id=area, category_id=category, view=view,
                    lookback_days=days if view == "historical" else 0,
                    start_date=str(start.date()), end_date=str(end.date()),
                    total_units=float(series.units.sum()) if complete else np.nan,
                    coverage_status="complete" if complete else "unavailable_incomplete_coverage",
                    observed_days=observed, expected_days=days, category_count=int(count)))
    return pd.DataFrame(rows, columns=columns)


In [ ]:
store_panel = daily_panel(transactions, coverage, cutoff, as_of, blocked_groups, STORE_GROUP)
store_statuses = eligibility(store_panel, cutoff, STORE_GROUP)
store_history = eligible_panel(store_panel, store_statuses, STORE_GROUP)
store_result = None
store_estimator = None
store_method = None
saved_store_bundle = None
store_evaluation = {"mode":MODE, "message":"Store model unavailable.", "limitations":evaluation["limitations"]}
if MODE == "train" and not store_history.empty:
    try:
        store_result = evaluate_scope(store_panel, transactions, coverage, blocked_groups, cutoff,
                                      STORE_GROUP, evaluation["limitations"])
    except EvaluationEvidenceError as error:
        # Lack of store evaluation evidence must not discard a valid barangay run.
        store_evaluation["message"] = str(error)
        store_statuses.loc[store_statuses.status.eq("eligible"), "status"] = "insufficient_store_evaluation"
    else:
        store_evaluation = store_result["evaluation"]
        store_method = store_result["published_method"]
        store_estimator = fit_method(store_method, store_history, STORE_GROUP)
elif MODE == "forecast_only" and SAVED_STORE_MODEL_PATH is not None:
    saved_store_bundle = joblib.load(choose_input(SAVED_STORE_MODEL_PATH))
    if (saved_store_bundle.get("scope") != "store" or saved_store_bundle["feature_version"] != FEATURE_VERSION
            or saved_store_bundle["data_kind"] != DATA_KIND
            or saved_store_bundle["cleaning_config"] != config
            or cutoff < pd.Timestamp(saved_store_bundle["model_training_end"])):
        raise ValueError("Incompatible saved store model, mappings, or forecast cutoff.")
    store_statuses = restrict_saved_groups(store_statuses, saved_store_bundle, STORE_GROUP)
    store_method = saved_store_bundle["method"]
    store_estimator = saved_store_bundle["estimator"]
    store_evaluation = {"mode":MODE, "training_evaluation":saved_store_bundle["evaluation"],
                        "model_training_end":saved_store_bundle["model_training_end"],
                        "message":"Using saved store model; no new accuracy measurement."}
elif MODE == "forecast_only":
    store_statuses.loc[store_statuses.status.eq("eligible"), "status"] = "saved_store_model_unavailable"

store_history = eligible_panel(store_panel, store_statuses, STORE_GROUP)
store_forecast = predict_seven(store_method, store_estimator, store_history, cutoff, STORE_GROUP)
store_forecast["lower_bound"] = None
store_forecast["upper_bound"] = None
store_forecast["unit_policy"] = "recorded_sale_units"
area_candidates = store_candidates = pd.DataFrame()
if MODE == "train":
    area_candidates = candidate_forecasts(history, cutoff, GROUP, published_method, estimator)
    if store_method is not None:
        store_candidates = candidate_forecasts(store_history, cutoff, STORE_GROUP, store_method, store_estimator)
else:
    print("Candidate comparison charts require MODE='train'; published forecasts use saved models.")

if COORDINATES_PATH is not None:
    COORDINATES_PATH = choose_input(COORDINATES_PATH)
    coordinates = prepare_coordinates(pd.read_csv(COORDINATES_PATH, dtype="string", keep_default_na=False))
elif DATA_KIND == "synthetic":
    coordinates = demo_coordinates(panel.area_id.unique())
else:
    coordinates = prepare_coordinates(pd.DataFrame(columns=["area_id", "latitude", "longitude", "coordinate_source"]))
map_summary = map_summaries(panel, forecast, cutoff)
print("Individual-store results are separate from area totals; store forecasts need not add up to the area forecast.")
print(forecast_explanation(store_forecast, store_method, globals().get("area_labels"), store=True))
display(readable_table(store_statuses, globals().get("area_labels")))


## Explain the result in plain language

This report uses measured results rather than invented explanations. Feature importance,
when available, describes model usage, not causal effects. Percentage changes require a
positive, complete previous week from the same cohort. Compare areas within a category;
different categories and package mixes do not imply equivalent physical consumption.


In [ ]:
def explain_run(forecast, observations, statuses, audit, evaluation, method, cutoff, data_kind, labels=None):
    lines = [f"# Forecast run report ({data_kind.upper()})", "",
        f"Estimate recorded sales from {(cutoff+pd.Timedelta(days=1)).date()} through {(cutoff+pd.Timedelta(days=7)).date()}, using history through {cutoff.date()}.",
        "This helps review expected category sales across participating stores. It does not measure all resident demand or prescribe inventory purchases.",
        "## What data was usable?", cleaning_explanation(audit),
        f"{int(statuses.status.eq('eligible').sum())} area/category combinations qualify; {int(statuses.status.ne('eligible').sum())} are withheld. A combination is one area and one product category.",
        "A recorded unit is one sellable package/item. Packs and bottles are not converted into equivalent contents.",
        "## Which method will be used, and why?", METRIC_GUIDE, evaluation_explanation(evaluation, method),
        "## What do the forecasts mean?", FORECAST_GUIDE, forecast_explanation(forecast, method, labels)]
    for key, g in forecast.groupby(GROUP):
        area = (labels or {}).get(key[0], key[0])
        past = observations.loc[observations.area_id.eq(key[0]) & observations.category_id.eq(key[1]) &
                                observations.date.between(cutoff-pd.Timedelta(days=6), cutoff)]
        total = float(g.predicted_units.sum())
        comparison = "No complete positive previous week is available for a percentage comparison."
        if len(past) == 7 and past.units_sold.notna().all() and past.units_sold.sum() > 0:
            change = (total/past.units_sold.sum()-1)*100
            comparison = f"{abs(change):.1f}% {'higher' if change >= 0 else 'lower'} than the previous complete week's recorded sales."
        lines.append(f"- {area} / {category_label(key[1])}: {total:.1f} recorded units expected over seven days. {comparison}")
    lines += ["## What can and cannot be concluded?",
        "These changes describe the forecast, not the reason sales change. Check your sales, stock and reporting records before deciding on an explanation.",
        "Current final sale statuses and location/category mappings may differ from what was recorded historically; this replay cannot reconstruct missing correction history.",
        "Stockouts can suppress recorded sales; the forecast assumes the participating stores remain the same. Reporting completeness is supplied or assumed, not independently verified.",
        "No estimated low-to-high prediction range is calculated; no 90% coverage claim is made.",
        "Synthetic scores demonstrate this workflow, not accuracy on real customers." if data_kind == "synthetic" else
        "Agree on acceptable business error and evaluate new real outcomes before relying on the forecast."]
    return "\n\n".join(lines) + "\n"

report = explain_run(forecast, panel, statuses, cleaning_audit, evaluation, published_method, cutoff, DATA_KIND, area_labels)
show_markdown(report)

import matplotlib.pyplot as plt
figures = []
for key, g in forecast.groupby(GROUP):
    if len(figures) >= 3:
        break
    actual = panel.loc[panel.area_id.eq(key[0]) & panel.category_id.eq(key[1]) &
                       panel.date.gt(cutoff-pd.Timedelta(days=28))]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(actual.date, actual.units_sold, label="Recorded sales (gaps = unknown)")
    ax.plot(g.forecast_date, g.predicted_units, label="Seven-day estimate", linestyle="--")
    ax.axvline(cutoff + pd.Timedelta(hours=12), color="gray", linestyle=":")
    ax.set(title=f"{DATA_KIND.upper()} | {area_labels.get(key[0], key[0])} / {category_label(key[1])} | {method_label(published_method)}", ylabel="Recorded sale units", xlabel="Date")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    describe_figure(fig, "Solid line: recorded daily sales over the past 28 days. Dashed line: the next seven days' estimates. The vertical line separates history from the future. Look for recurring peaks and whether estimates follow recent levels; future actual sales are not yet known.",
        daily_chart_takeaway(actual.rename(columns={"units_sold":"units"}), g.rename(columns={"forecast_date":"date", "predicted_units":"units"}), cutoff))
    figures.append(fig)
    show_guided_figure(fig)
print("How the selected model uses its inputs: recent averages describe the usual sales level; lagged sales describe specific earlier dates; variation describes how much sales fluctuate. Calendar and identity inputs allow different date/place/category patterns.")
feature_table, feature_text = feature_usage(estimator, published_method)
if not feature_table.empty:
    display(feature_table)
print(feature_text)
report += "\n## How the selected model uses its inputs\n\n" + feature_text + "\n"
for row in feature_table.itertuples(index=False, name=None):
    report += f"- {row[0]}: {row[1]:g}\n"


## Demand preview and notebook-only model diagnostics
Historical Demand uses the past 90/180 dates; Forecast Demand always predicts the next
seven dates. Category defaults to All categories (mixed recorded sale units).
The map summarizes barangays. Select Individual store below it for a store trend.
Filters redraw results without fitting models. Experiment charts are not seller UI.


In [ ]:
def comparison_figures(predictions, comparison, candidates, observations, published, published_method,
                       area_id, category_id, cutoff, data_kind, area_labels, store_id=None):
    import matplotlib.pyplot as plt
    figures = []
    if predictions.empty or candidates.empty:
        return figures
    def subset(frame):
        result = frame.loc[frame.area_id.eq(area_id)].copy()
        if store_id is not None:
            result = result.loc[result.store_id.eq(store_id)]
        if category_id != ALL_CATEGORIES:
            result = result.loc[result.category_id.eq(category_id)]
        return result
    p = subset(predictions)
    scope = f"Store {store_id}" if store_id is not None else "Barangay"
    category_title = "All categories (mixed sale units)" if category_id == ALL_CATEGORIES else category_label(category_id)
    title = f"{data_kind.upper()} | {scope} | {area_labels.get(area_id, area_id)} | {category_title}"
    fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
    expected_categories = (observations.loc[observations.area_id.eq(area_id) &
        (observations.store_id.eq(store_id) if store_id is not None else True)].category_id.nunique()
        if category_id == ALL_CATEGORIES else 1)
    for method, ax in zip(TREE_METHODS, axes):
        g = p.loc[p.method.eq(method)]
        dates, actual, predicted = [], [], []
        for date, d in g.groupby("forecast_date", sort=True):
            complete = len(d) == expected_categories and d.category_id.nunique() == expected_categories
            dates.append(date)
            actual.append(d.actual_units.sum() if complete else np.nan)
            predicted.append(d.predicted_units.sum() if complete else np.nan)
        ax.plot(dates, actual, label="Actual recorded sales")
        ax.plot(dates, predicted, "--", label=method_label(method))
        ax.set_ylabel("Recorded units")
        ax.legend(loc="upper left")
    fig.suptitle(title + "\nValidation actual vs predicted (seven-day issuance windows)")
    axes[-1].set_xlabel("Date")
    fig.autofmt_xdate()
    fig.tight_layout()
    local = compare_methods(p.loc[p.method.isin(TREE_METHODS)])
    local_text = "; ".join(f"{method_label(r.method)}: average daily error {r.MAE:.2f} units" for r in local.itertuples())
    describe_figure(fig, "These are historical checks for the selected area/category or store. Each panel compares one model with the same actual recorded sales. Closer lines mean smaller misses; estimates above actuals are too high and below actuals too low. For All categories, only complete category totals are drawn. These are past forecasts, not fitted training lines.",
        (local_text + ". " + method_label(local.sort_values("MAE").iloc[0].method) + " has the lowest MAE for these selected category/date rows." if not local.empty else "No comparable historical points are available for this selection.") + " Scores here are per category/date; All categories lines are complete daily sums. These selected-series errors need not match the overall method ranking.")
    figures.append(fig)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for ax, metric in zip(axes, ["MAE", "RMSE"]):
        ordered = comparison.sort_values(metric)
        ax.barh(ordered.method.map(method_chart_label), ordered[metric])
        ax.set_title("Validation " + metric + " — all eligible series in this scope")
        ax.set_xlabel("Daily error in units; lower is better")
    fig.suptitle(f"{data_kind.upper()} | {scope} | Overall method comparison")
    fig.tight_layout()
    describe_figure(fig, "These bars compare all eligible combinations in this scope, even when the line chart shows one selection. MAE is average absolute daily error; RMSE gives bigger misses more weight. Shorter bars are better. The notebook chooses by MAE, not by a vote between measures.", validation_explanation(comparison))
    figures.append(fig)
    actual, _ = selected_daily(observations, published, area_id, category_id, store_id)
    actual = actual.loc[actual.date.gt(cutoff-pd.Timedelta(days=28))]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(actual.date, actual.units, color="black", label="Recorded sales")
    for method in TREE_METHODS:
        _, future = selected_daily(observations, candidates.loc[candidates.method.eq(method)],
                                   area_id, category_id, store_id)
        ax.plot(future.date, future.units, "--", label=method_label(method))
    if published_method in BASELINES:
        _, future = selected_daily(observations, published, area_id, category_id, store_id)
        ax.plot(future.date, future.units, "--", label=method_chart_label(published_method) + " (used for forecast)")
    ax.axvline(cutoff+pd.Timedelta(hours=12), color="gray", linestyle=":")
    ax.set(title=title + f"\nNext seven days | method used: {method_label(published_method)}", ylabel="Recorded sale units", xlabel="Date")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    _, used_future = selected_daily(observations, published, area_id, category_id, store_id)
    describe_figure(fig, "The solid line is recorded history; dashed lines are different models estimating the same next seven days. The vertical line marks where the future begins. Agreement shows similar estimates, not proven accuracy. The title names the method actually used; other lines are experiments.", daily_chart_takeaway(actual, used_future, cutoff))
    figures.append(fig)
    return figures

def demand_map(summaries, coordinates, labels, category_id, view, days, selected_area, data_kind):
    import folium
    from folium.plugins import HeatMap
    from html import escape
    rows = summaries.loc[summaries.category_id.eq(category_id) & summaries.view.eq(view)]
    if view == "historical":
        rows = rows.loc[rows.lookback_days.eq(days)]
    rows = rows.copy()
    rows["barangay"] = rows.area_id.map(lambda a: labels.get(a, a))
    category_title = "All categories (mixed sale units)" if category_id == ALL_CATEGORIES else category_label(category_id)
    rows = rows.merge(coordinates, on="area_id", how="left", validate="many_to_one")
    placed = rows.loc[rows.latitude.notna() & rows.longitude.notna()]
    center = [float(placed.latitude.mean()), float(placed.longitude.mean())] if len(placed) else [0, 0]
    m = folium.Map(location=center, zoom_start=12 if len(placed) else 2)
    if len(placed) > 1:
        m.fit_bounds(placed[["latitude", "longitude"]].values.tolist(), padding=(30, 30))
    heat = placed.loc[placed.total_units.notna() & placed.total_units.gt(0)]
    if len(heat):
        weights = heat[["latitude", "longitude", "total_units"]].astype(float).copy()
        weights["total_units"] /= weights.total_units.max()
        HeatMap(weights.values.tolist(),
                min_opacity=.25, radius=32, blur=22, max_zoom=15).add_to(m)
    for row in placed.itertuples():
        value = f"{row.total_units:,.1f}" if pd.notna(row.total_units) else "Unavailable"
        tooltip = (f"{labels.get(row.area_id, row.area_id)} | {category_title} | {value} recorded units | "
                   f"{row.start_date} to {row.end_date} | {row.observed_days}/{row.expected_days} complete days | "
                   f"{row.coverage_status} | {row.coordinate_source}")
        folium.Marker([row.latitude, row.longitude], icon=folium.DivIcon(
            html='<div style="background:white;border:1px solid #64748b;border-radius:4px;padding:2px 5px;white-space:nowrap;font-weight:600">'+escape(value)+'</div>',
            icon_anchor=(0, -10)), tooltip=escape(tooltip)).add_to(m)
        folium.CircleMarker([row.latitude, row.longitude], radius=8,
            color="#111827" if row.area_id == selected_area else "#2563eb",
            weight=4 if row.area_id == selected_area else 2, fill=True,
            fill_color="#6b7280" if pd.isna(row.total_units) else "#2563eb",
            tooltip=escape(tooltip), popup=escape(tooltip)).add_to(m)
    maximum = float(heat.total_units.max()) if len(heat) else 0.
    legend = (f"{data_kind.upper()} | Heat: lower to higher recorded sale units (0–{maximum:,.1f}). "
              "Numeric totals are in marker tooltips and the table. Heat is visual smoothing, not street-level demand.")
    if category_id == ALL_CATEGORIES:
        legend += " Mixed sale units: packs, bottles, etc."
    if coordinates.coordinate_source.astype(str).str.startswith("synthetic_").any():
        legend += " Illustrative coordinates; not authoritative barangay centroids."
    elif len(coordinates):
        legend += " Sourced reference points; not store locations or official centroids."
    m.get_root().html.add_child(folium.Element(
        '<div style="position:fixed;bottom:12px;left:12px;z-index:9999;background:white;'
        'padding:10px;max-width:500px;border:1px solid #aaa">'+'<div style="height:8px;background:linear-gradient(to right,blue,cyan,lime,yellow,red);margin-bottom:6px"></div>'+escape(legend)+'</div>'))
    return m, rows

def dashboard_preview(observations, forecasts, store_observations, store_forecasts, summaries,
                      coordinates, labels, cutoff, data_kind, area_method, store_method,
                      validation, comparison, candidates, store_result, store_candidates):
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import display, HTML, clear_output
    from html import escape
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    areas = sorted(observations.area_id.unique())
    period = widgets.Dropdown(options=[90, 180], value=90, description="Past days:")
    category = widgets.Dropdown(options=[("All categories", ALL_CATEGORIES)] +
        [(category_label(c), c) for c in sorted(observations.category_id.unique())], description="Category:")
    area = widgets.Dropdown(options=[(labels.get(a, a), a) for a in areas], description="Barangay:")
    scope = widgets.ToggleButtons(options=[("Barangay", "area"), ("Individual store", "store")], value="area")
    store = widgets.Dropdown(options=[], description="Store:", disabled=True)
    tabs = widgets.Tab(children=[widgets.Output(), widgets.Output()])
    tabs.set_title(0, "Historical Demand")
    tabs.set_title(1, "Forecast Demand")
    diagnostics = widgets.Output()
    comparison_toggle = widgets.Checkbox(value=False, description="Show experiment model comparisons", indent=False)
    busy = False
    def refresh(change=None):
        nonlocal busy
        if busy:
            return
        busy = True
        try:
            available = sorted(store_observations.loc[store_observations.area_id.eq(area.value)].store_id.unique())
            previous_store = store.value
            store.options = available
            store.value = previous_store if previous_store in available else (available[0] if available else None)
            store.disabled = scope.value != "store" or not available
            view = "forecast" if tabs.selected_index == 1 else "historical"
            with tabs.children[tabs.selected_index]:
                clear_output(wait=True)
                display(HTML(f"<b>{escape(data_kind.upper())}</b> | Data through {cutoff.date()}. "
                    "Recorded participating-store sales; not total resident demand."))
                if category.value == ALL_CATEGORIES:
                    display(HTML("Mixed sale units: packs, bottles, etc."))
                map_object, table = demand_map(summaries, coordinates, labels, category.value, view,
                                               period.value, area.value, data_kind)
                print("How to read this map: compare totals for the same period and category. Historical Demand shows recorded sales; Forecast Demand shows estimates. More intense heat means higher totals, not more demand per resident or a street-level location. Read marker totals and coverage in the table; missing totals are unknown.")
                display(map_object)
                complete_totals = table.loc[table.total_units.notna()]
                if not complete_totals.empty:
                    largest = complete_totals.loc[complete_totals.total_units.idxmax()]
                    print(f"Map summary: {len(complete_totals)} areas have complete totals; {largest.barangay} has the largest shown total ({largest.total_units:.1f} recorded units). Totals reflect the participating stores, not population-adjusted demand.")
                else:
                    print("Map summary: no complete totals are available for this selection.")
                display(table[["barangay", "total_units", "start_date", "end_date", "coverage_status",
                               "observed_days", "expected_days", "coordinate_source"]])
                missing = int((table.latitude.isna() | table.longitude.isna()).sum())
                if missing:
                    display(HTML(f"{missing} area(s) have no supplied coordinates; their analytical totals remain in the table."))
                own = scope.value == "store"
                obs, future = (store_observations, store_forecasts) if own else (observations, forecasts)
                actual, predicted = selected_daily(obs, future, area.value, category.value, store.value if own else None)
                dates = pd.date_range(cutoff-pd.Timedelta(days=period.value-1), cutoff)
                actual = actual.set_index("date").reindex(dates)
                method = store_method if own else area_method
                fig, ax = plt.subplots(figsize=(11, 4))
                ax.plot(actual.index, actual.units, label="Recorded sales (gaps = unknown)")
                if view == "forecast":
                    ax.plot(predicted.date, predicted.units, "--", label=f"Seven-day forecast: {method or 'unavailable'}")
                    ax.axvline(cutoff+pd.Timedelta(hours=12), color="gray", linestyle=":")
                    if predicted.empty or predicted.units.isna().any():
                        display(HTML("Forecast unavailable for one or more selected series; missing values are not zero."))
                title = f"Store {store.value}" if own else labels.get(area.value, area.value)
                category_title = "All categories (mixed sale units)" if category.value == ALL_CATEGORIES else category_label(category.value)
                ax.set(title=f"{title} | {category_title} | {period.value} days of history", ylabel="Recorded sale units", xlabel="Date")
                ax.legend()
                fig.autofmt_xdate()
                fig.tight_layout()
                describe_figure(fig, "The horizontal axis is date; the vertical axis counts recorded sale units. The solid line is known daily sales, with unknown reporting shown as gaps. " + ("The dashed line estimates the next seven days and the vertical line separates history from the future." if view == "forecast" else "Look for repeated weekday peaks or changes in sales levels; no future sales are shown in this view."),
                    daily_chart_takeaway(actual.rename_axis("date").reset_index(), predicted if view == "forecast" else None, cutoff))
                show_guided_figure(fig)
                display(actual.rename_axis("date").reset_index())
                if view == "forecast":
                    display(predicted)
                    display(HTML("Store forecasts are independently modeled and need not sum to barangay forecasts."))
            with diagnostics:
                clear_output(wait=True)
                if comparison_toggle.value:
                    result = store_result if scope.value == "store" else None
                    v = result["validation_predictions"] if result is not None else validation
                    c = result["comparison"] if result is not None else comparison
                    candidate = store_candidates if scope.value == "store" else candidates
                    if scope.value == "store" and result is None:
                        display(HTML("Store candidate comparisons require a successful training run."))
                    else:
                        diagnostic_figures = comparison_figures(v, c, candidate, obs, future, method,
                            area.value, category.value, cutoff, data_kind, labels, store.value if own else None)
                        if not diagnostic_figures:
                            display(HTML("Candidate comparison charts require MODE='train'."))
                        for fig in diagnostic_figures:
                            show_guided_figure(fig)
        finally:
            busy = False
    for control in [period, category, area, scope, store, comparison_toggle]:
        control.observe(refresh, names="value")
    tabs.observe(refresh, names="selected_index")
    root = widgets.VBox([widgets.HBox([period, category]), area, scope, store, tabs,
                         widgets.HTML("<b>Notebook experiment diagnostics — not seller-facing</b>"),
                         comparison_toggle, diagnostics])
    display(root)
    refresh()
    return {"widget":root, "period":period, "category":category, "area":area, "scope":scope,
            "store":store, "tabs":tabs, "comparisons":comparison_toggle, "refresh":refresh}


In [ ]:
# Export representative diagnostics for both scopes; the widget supports every series.
if MODE == "train":
    for obs, published, candidate, result, method, keys in [
        (panel, forecast, area_candidates, area_result, published_method, GROUP),
        (store_panel, store_forecast, store_candidates, store_result, store_method, STORE_GROUP)]:
        if result is None or candidate.empty:
            continue
        first = candidate.iloc[0]
        diagnostic = comparison_figures(result["validation_predictions"], result["comparison"], candidate,
            obs, published, method, first.area_id, first.category_id, cutoff, DATA_KIND, area_labels,
            first.store_id if "store_id" in keys else None)
        for fig in diagnostic:
            figures.append(fig)
            show_guided_figure(fig)
if SHOW_DASHBOARD:
    preview = dashboard_preview(panel, forecast, store_panel, store_forecast, map_summary, coordinates,
        area_labels, cutoff, DATA_KIND, published_method, store_method, validation_predictions, comparison,
        area_candidates, store_result, store_candidates)


## Your complete result, before export

Read this section for the dataset, chosen method, error tradeoffs, forecast highlights, and practical limits in one place. The same summary is included in the downloaded run report.


In [ ]:
summary = run_summary(transactions, panel, statuses, forecast, cleaning_audit, evaluation, published_method, cutoff, DATA_KIND, estimator, store_evaluation, store_method, area_labels)
show_markdown(summary)
report += "\n" + summary


## Versioned export and trusted model reload

Backend handoff: structured forecast CSV, historical observations, statuses, evaluation,
and metadata. The application importer should not load Python serialization. Keep
`method.joblib` and historical state for trusted Python inference only. A baseline winner
is valid: the saved bundle then stores its method/configuration with no fitted estimator.

Download both the results ZIP and notebook before the Colab session expires. Raw rows,
quarantine records, store cohorts, and serialized models are operator artifacts, not public API data.


In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
run_dir = OUTPUT_ROOT/run_id
run_dir.mkdir(parents=True, exist_ok=False)
forecast["run_id"] = run_id
forecast.to_csv(run_dir/"daily_future_forecast.csv", index=False)
panel.to_csv(run_dir/"historical_observations.csv", index=False)
statuses.to_csv(run_dir/"coverage_summary.csv", index=False)
quarantine.to_csv(run_dir/"quarantine_private.csv", index=False)
duplicates.to_csv(run_dir/"duplicates_private.csv", index=False)
json_write(run_dir/"cleaning_summary.json", cleaning_audit)
json_write(run_dir/"evaluation_summary.json", evaluation)
(run_dir/"run_report.md").write_text(report, encoding="utf-8")
transactions[["area_id", "city", "barangay"]].drop_duplicates().to_csv(run_dir/"area_labels.csv", index=False)
normalization_map = raw[["city", "barangay"]].drop_duplicates().copy()
normalization_map[["canonical_city", "canonical_barangay", "area_id"]] = [
    canonical_area(c, b, config) for c, b in zip(normalization_map.city, normalization_map.barangay)]
normalization_map.to_csv(run_dir/"area_normalization_map.csv", index=False)
transactions.groupby(["category_id", "unit_label"], as_index=False).quantity.sum().to_csv(run_dir/"unit_mix.csv", index=False)
history.groupby(GROUP, group_keys=False).tail(28).to_csv(run_dir/"history_state.csv", index=False)
if MODE == "train":
    comparison.drop(columns="tie_priority").to_csv(run_dir/"model_comparison.csv", index=False)
    all_backtests = pd.concat([validation_predictions.assign(stage="validation"), final_predictions.assign(stage="final")], ignore_index=True)
    all_backtests.to_csv(run_dir/"backtest_predictions.csv", index=False)
    evaluation_refusals.to_csv(run_dir/"evaluation_refusals.csv", index=False)
    breakdowns = []
    for dimensions in [["method", "horizon_day"], ["method", *GROUP]]:
        for key, g in final_predictions.groupby(dimensions):
            breakdowns.append({"dimension":"/".join(dimensions), **dict(zip(dimensions, key)), **metrics(g)})
    json_write(run_dir/"final_metric_breakdowns.json", breakdowns)
    weekly = final_predictions.groupby(["method", *GROUP, "cutoff"], as_index=False).agg(
        actual_units=("actual_units", "sum"), predicted_units=("predicted_units", "sum"), days=("forecast_date", "nunique"))
    weekly = weekly.loc[weekly.days.eq(7)]
    json_write(run_dir/"weekly_total_metrics.json", compare_methods(weekly).to_dict("records"))

versions = {name:importlib.metadata.version(name) for name in
            ["numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "joblib", "matplotlib", "ipywidgets", "folium"]}
source_files = {"transactions.csv":sha256_file(CSV_PATH)}
if not coverage_assumed:
    source_files["coverage.csv"] = sha256_file(COVERAGE_PATH)
dataset_hash = hashlib.sha256(json.dumps({"files":source_files, "config":config,
    "assumption": {"enabled":coverage_assumed, "start":REPORTING_START,
        "end":REPORTING_END, "cohort":DECLARED_COHORT}}, sort_keys=True).encode()).hexdigest()
if MODE == "train":
    bundle = {"schema_version":SCHEMA_VERSION, "feature_version":FEATURE_VERSION,
        "method":published_method, "estimator":estimator, "cleaning_config":config,
        "supported_groups":history[GROUP].drop_duplicates().values.tolist(),
        "cohorts":{json.dumps((r.area_id, r.category_id)):r.cohort_id for r in statuses.itertuples() if r.status == "eligible"},
        "model_training_end":str(cutoff.date()), "training_dataset_hash":dataset_hash,
        "data_kind":DATA_KIND, "evaluation":evaluation, "dependency_versions":versions}
else:
    bundle = saved_bundle
joblib.dump(bundle, run_dir/"method.joblib", compress=3)
reloaded = joblib.load(run_dir/"method.joblib")
parity = predict_seven(reloaded["method"], reloaded["estimator"], history, cutoff)
if not np.allclose(parity.predicted_units, forecast.predicted_units, rtol=1e-10, atol=1e-10):
    raise AssertionError("Saved-method prediction parity failed.")
for i, fig in enumerate(figures):
    fig.savefig(run_dir/f"forecast_preview_{i+1}.png", dpi=120)


# Additive dashboard and private store artifacts; primary area CSV columns are unchanged.
coordinates.to_csv(run_dir/"area_coordinates.csv", index=False)
map_summary.assign(run_id=run_id).to_csv(run_dir/"map_summaries.csv", index=False)
store_panel.to_csv(run_dir/"store_historical_observations.csv", index=False)
store_statuses.to_csv(run_dir/"store_coverage_summary.csv", index=False)
store_forecast["run_id"] = run_id
store_forecast.to_csv(run_dir/"store_daily_future_forecast.csv", index=False)
store_history.groupby(STORE_GROUP, group_keys=False).tail(28).to_csv(run_dir/"store_history_state.csv", index=False)
json_write(run_dir/"store_evaluation_summary.json", store_evaluation)
if MODE == "train":
    area_candidates.to_csv(run_dir/"candidate_future_forecasts.csv", index=False)
    if store_result is not None:
        store_result["comparison"].drop(columns="tie_priority").to_csv(run_dir/"store_model_comparison.csv", index=False)
        pd.concat([store_result["validation_predictions"].assign(stage="validation"),
                   store_result["final_predictions"].assign(stage="final")], ignore_index=True).to_csv(
                       run_dir/"store_backtest_predictions.csv", index=False)
        store_result["refusals"].to_csv(run_dir/"store_evaluation_refusals.csv", index=False)
        store_candidates.to_csv(run_dir/"store_candidate_future_forecasts.csv", index=False)
store_bundle = saved_store_bundle
if MODE == "train" and store_method is not None:
    store_bundle = {"schema_version":SCHEMA_VERSION, "feature_version":FEATURE_VERSION, "scope":"store",
        "method":store_method, "estimator":store_estimator, "cleaning_config":config,
        "supported_groups":store_history[STORE_GROUP].drop_duplicates().values.tolist(),
        "cohorts":{json.dumps(tuple(getattr(r, k) for k in STORE_GROUP)):r.cohort_id
                   for r in store_statuses.itertuples() if r.status == "eligible"},
        "model_training_end":str(cutoff.date()), "training_dataset_hash":dataset_hash,
        "data_kind":DATA_KIND, "evaluation":store_evaluation, "dependency_versions":versions}
if store_bundle is not None:
    joblib.dump(store_bundle, run_dir/"store_method.joblib", compress=3)
    reloaded_store = joblib.load(run_dir/"store_method.joblib")
    store_parity = predict_seven(reloaded_store["method"], reloaded_store["estimator"], store_history, cutoff, STORE_GROUP)
    if not np.allclose(store_parity.predicted_units, store_forecast.predicted_units, rtol=1e-10, atol=1e-10):
        raise AssertionError("Saved store model prediction parity failed.")

# Copy the executing notebook when it is accessible locally. Colab users also save
# their edited notebook from File > Download, because the runtime may not expose it.
notebook_paths = list(Path.cwd().glob("*.ipynb"))
notebook_hash = None
if len(notebook_paths) == 1:
    notebook_hash = sha256_file(notebook_paths[0])
    (run_dir/"training_notebook.ipynb").write_bytes(notebook_paths[0].read_bytes())
else:
    print("Save the edited Colab notebook separately to preserve executable source.")
metadata = {"schema_version":SCHEMA_VERSION, "feature_version":FEATURE_VERSION,
    "run_id":run_id, "data_kind":DATA_KIND, "target":TARGET_ID, "unit_policy":"recorded_sale_units",
    "method":published_method, "model_training_end":bundle["model_training_end"],
    "forecast_cutoff":str(cutoff.date()), "forecast_start":str((cutoff+pd.Timedelta(days=1)).date()),
    "forecast_end":str((cutoff+pd.Timedelta(days=7)).date()), "forecast_days":7,
    "as_of":str(as_of), "generated_at":datetime.now(timezone.utc).isoformat(),
    "dataset_hash":dataset_hash, "training_dataset_hash":bundle["training_dataset_hash"],
    "source_files":source_files, "cleaning_config":config, "coverage_assumed":coverage_assumed,
    "reporting_assumptions":{"start":REPORTING_START, "end":REPORTING_END, "cohort":DECLARED_COHORT} if coverage_assumed else None,
    "smoke_run":SMOKE_RUN, "model_trees":MODEL_TREES, "random_seed":RANDOM_SEED,
    "dependency_versions":versions, "training_dependency_versions":bundle["dependency_versions"],
    "notebook_sha256":notebook_hash, "intervals":"unavailable", "reload_parity":"passed",
    "refused_groups":statuses.loc[statuses.status.ne("eligible")].to_dict("records")}
# Fingerprint executed pipeline code even if Colab does not expose its notebook file.
import marshal
metadata["executed_code_sha256"] = hashlib.sha256(b"".join(marshal.dumps(fn.__code__) for fn in
    [canonical_area, clean_transactions, prepare_coverage, daily_panel, eligibility,
     feature_row, training_examples, build_estimator, predict_seven, backtest, evaluate_scope,
     selected_daily, map_summaries, prepare_coordinates])).hexdigest()
metadata["dashboard_version"] = DASHBOARD_VERSION
metadata["store_method"] = store_method
metadata["store_model_training_end"] = store_bundle["model_training_end"] if store_bundle else None
metadata["store_training_dataset_hash"] = store_bundle["training_dataset_hash"] if store_bundle else None
metadata["coordinate_source_sha256"] = sha256_file(COORDINATES_PATH) if COORDINATES_PATH else None
metadata["private_artifacts"] = [p.name for p in run_dir.iterdir() if p.name.startswith("store_")]
metadata["artifacts"] = {p.name:sha256_file(p) for p in sorted(run_dir.iterdir()) if p.is_file()}
json_write(run_dir/"run_metadata.json", metadata)

def verify_bundle(directory):
    directory = Path(directory)
    meta = json.loads((directory/"run_metadata.json").read_text(encoding="utf-8"))
    for name, checksum in meta["artifacts"].items():
        if Path(name).name != name or sha256_file(directory/name) != checksum:
            raise ValueError("Artifact checksum/path mismatch: " + name)
    f = pd.read_csv(directory/"daily_future_forecast.csv", dtype={"area_id":str, "category_id":str})
    status_table = pd.read_csv(directory/"coverage_summary.csv", dtype={"area_id":str, "category_id":str})
    expected_groups = set(map(tuple, status_table.loc[status_table.status.eq("eligible"), GROUP].to_numpy()))
    actual_groups = set(map(tuple, f[GROUP].drop_duplicates().to_numpy()))
    if actual_groups != expected_groups:
        raise ValueError("Forecast groups do not match eligible coverage statuses.")
    if not f.empty and (f.duplicated([*GROUP, "forecast_date"]).any() or not np.isfinite(f.predicted_units).all() or f.predicted_units.lt(0).any()):
        raise ValueError("Invalid forecast values or duplicate keys.")
    expected_dates = set(pd.date_range(meta["forecast_start"], meta["forecast_end"]).strftime("%Y-%m-%d"))
    for _, g in f.groupby(GROUP):
        if len(g) != 7 or set(g.forecast_date) != expected_dates:
            raise ValueError("Incomplete forecast sequence.")
    if not f.empty and (not f.run_id.eq(meta["run_id"]).all() or not f.unit_policy.eq("recorded_sale_units").all()):
        raise ValueError("Forecast metadata mismatch.")

    if "dashboard_version" in meta:
        if meta["dashboard_version"] != DASHBOARD_VERSION:
            raise ValueError("Unsupported dashboard extension version.")
        for name in ["area_coordinates.csv", "map_summaries.csv", "store_historical_observations.csv",
                     "store_coverage_summary.csv", "store_daily_future_forecast.csv", "store_evaluation_summary.json"]:
            if name not in meta["artifacts"]:
                raise ValueError("Missing declared dashboard artifact: " + name)
        prepare_coordinates(pd.read_csv(directory/"area_coordinates.csv", dtype="string", keep_default_na=False))
        sf = pd.read_csv(directory/"store_daily_future_forecast.csv", dtype={k:str for k in STORE_GROUP})
        ss = pd.read_csv(directory/"store_coverage_summary.csv", dtype={k:str for k in STORE_GROUP})
        expected = set(map(tuple, ss.loc[ss.status.eq("eligible"), STORE_GROUP].to_numpy()))
        actual = set(map(tuple, sf[STORE_GROUP].drop_duplicates().to_numpy()))
        if actual != expected or (expected and not meta.get("store_method")):
            raise ValueError("Store forecasts do not match eligible statuses/model.")
        if (sf.duplicated([*STORE_GROUP, "forecast_date"]).any() or not np.isfinite(sf.predicted_units).all()
                or sf.predicted_units.lt(0).any()):
            raise ValueError("Invalid store forecast values or duplicate keys.")
        if not sf.empty and (not sf.run_id.eq(meta["run_id"]).all() or not sf.unit_policy.eq("recorded_sale_units").all()):
            raise ValueError("Store forecast metadata mismatch.")
        for _, g in sf.groupby(STORE_GROUP):
            if len(g) != 7 or set(g.forecast_date) != expected_dates:
                raise ValueError("Incomplete store forecast sequence.")
        observations = pd.read_csv(directory/"historical_observations.csv", dtype={k:str for k in GROUP})
        observations["date"] = pd.to_datetime(observations.date)
        f["forecast_date"] = pd.to_datetime(f.forecast_date)
        summary = pd.read_csv(directory/"map_summaries.csv", dtype={"area_id":str, "category_id":str})
        if not summary.run_id.eq(meta["run_id"]).all():
            raise ValueError("Map run mismatch.")
        expected_summary = map_summaries(observations, f, pd.Timestamp(meta["forecast_cutoff"]))
        pd.testing.assert_frame_equal(summary.drop(columns="run_id"), expected_summary,
                                      check_dtype=False, rtol=1e-10, atol=1e-10)
    return True

verify_bundle(run_dir)
zip_path = OUTPUT_ROOT/f"{run_id}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.iterdir()):
        if path.name in metadata["artifacts"] or path.name == "run_metadata.json":
            archive.write(path, arcname=path.name)
print("Verified bundle:", run_dir)
print("Download:", zip_path)
# In Colab, uncomment:
# from google.colab import files
# files.download(str(zip_path))


## Next cycle

1. Hand the structured outputs and metadata to the backend repository's importer.
2. Keep the issued forecast immutable; collect actual sales and reporting evidence.
3. For a forecast refresh, set `MODE="forecast_only"`, point to the trusted saved
   `method.joblib`, upload newer history/coverage, and set the next complete cutoff.
   Include the entire relevant reporting history (at least 56 complete dates), not just new rows.
4. For weekly retraining, set `MODE="train"`, rerun comparison and final testing, then
   hand off a new versioned bundle. Reused final windows are monitoring evidence,
   not a new untouched test for repeated redesign.
5. Reconcile late data, review errors/coverage and new groups, and preserve previous
   bundles for rollback. Refreshing a forecast and retraining a model are different operations.

See `ML_WORKFLOW_GUIDE.md` for the complete first-run and recurring workflow.
